In [157]:
# Data Cleaning for Recipe Dataset
import pandas as pd
import numpy as np
import re
import os
import kagglehub as kagglehub

from difflib import SequenceMatcher
from collections import Counter


In [158]:
# Download dataset files to a local folder
path = kagglehub.dataset_download("irkaal/foodcom-recipes-and-reviews")

print("Dataset downloaded to:", path)
print("Files:", os.listdir(path))

# Load CSVs
recipes = pd.read_csv(os.path.join(path, "recipes.csv"))
reviews = pd.read_csv(os.path.join(path, "reviews.csv"))

print(recipes.head())
print(reviews.head())
print(recipes.shape, reviews.shape)

Dataset downloaded to: /Users/makabaka/.cache/kagglehub/datasets/irkaal/foodcom-recipes-and-reviews/versions/2
Files: ['reviews.csv', 'recipes.csv', 'recipes.parquet', 'reviews.parquet']
   RecipeId                               Name  AuthorId      AuthorName  \
0        38  Low-Fat Berry Blue Frozen Dessert      1533          Dancer   
1        39                            Biryani      1567        elly9812   
2        40                      Best Lemonade      1566  Stephen Little   
3        41     Carina's Tofu-Vegetable Kebabs      1586         Cyclopz   
4        42                       Cabbage Soup      1538       Duckie067   

  CookTime PrepTime TotalTime         DatePublished  \
0    PT24H    PT45M  PT24H45M  1999-08-09T21:46:00Z   
1    PT25M     PT4H   PT4H25M  1999-08-29T13:12:00Z   
2     PT5M    PT30M     PT35M  1999-09-05T19:52:00Z   
3    PT20M    PT24H  PT24H20M  1999-09-03T14:54:00Z   
4    PT30M    PT20M     PT50M  1999-09-19T06:19:00Z   

                         

In [159]:
pd.set_option('display.max_columns', None)

recipes.head()

,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,RecipeCategory,Keywords,RecipeIngredientQuantities,RecipeIngredientParts,AggregatedRating,ReviewCount,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg"", \n""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/picuaETeN.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/pictzvxW5.jpg"")",Frozen Desserts,"c(""Dessert"", ""Low Protein"", ""Low Cholesterol"", ""Healthy"", ""Free Of..."", ""Summer"", ""Weeknight"", ""Freezer"", ""Easy"")","c(""4"", ""1/4"", ""1"", ""1"")","c(""blueberries"", ""granulated sugar"", ""vanilla yogurt"", ""lemon juice"")",4.5,4.0,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,NaN,"c(""Toss 2 cups berries with sugar."", ""Let stand for 45 minutes, stirring occasionally."", ""Transfer berry-sugar mixture to food processor."", ""Add yogurt and process until smooth."", ""Strain through fine sieve. Pour into baking pan (or transfer to ice cream maker and process according to manufacturers' directions). Freeze uncovered until edges are solid but centre is soft. Transfer to processor and blend until smooth again."", ""Return to pan and freeze until edges are solid."", ""Transfer to processor and blend until smooth again."", \n""Fold in remaining 2 cups of blueberries."", ""Pour into plastic mold and freeze overnight. Let soften slightly to serve."")"
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/39/picM9Mhnw.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/39/picHv4Ocr.jpg"")",Chicken Breast,"c(""Chicken Thigh & Leg"", ""Chicken"", ""Poultry"", ""Meat"", ""Asian"", ""Indian"", ""Weeknight"", ""Stove Top"")","c(""1"", ""4"", ""2"", ""2"", ""8"", ""1/4"", ""8"", ""1/2"", ""1"", ""1"", ""1/4"", ""1/4"", ""1/2"", ""1/4"", ""2"", ""3"", NA, ""2"", ""1"", ""1"", ""8"", ""2"", ""1/3"", ""1/3"", ""1/3"", ""6"")","c(""saffron"", ""milk"", ""hot green chili peppers"", ""onions"", ""garlic"", ""clove"", ""peppercorns"", ""cardamom seed"", ""cumin seed"", ""poppy seed"", ""mace"", ""cilantro"", ""mint leaf"", ""fresh lemon juice"", ""plain yogurt"", ""boneless chicken"", ""salt"", ""ghee"", ""onion"", ""tomatoes"", ""basmati rice"", ""long-grain rice"", ""raisins"", ""cashews"", ""eggs"")",3.0,1.0,1110.7,58.8,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,NaN,"c(""Soak saffron in warm milk for 5 minutes and puree in blender."", ""Add chiles, onions, ginger, garlic, cloves, peppercorns, cardamom seeds, cinnamon, coriander and cumin seeds, poppy seeds, nutmeg, mace, cilantro or mint leaves and lemon juice. Blend into smooth paste. Put paste into large bowl, add yogurt and mix well."", ""Marinate chicken in yogurt mixture with salt, covered for at least 2 - 6 hours in refrigerator."", ""In skillet. heat oil over med

In [160]:
reviews.head()

,ReviewId,RecipeId,AuthorId,AuthorName,Rating,Review,DateSubmitted,DateModified
0,2,992,2008,gayg msft,5,better than any you can get at a restaurant!,2000-01-25T21:44:00Z,2000-01-25T21:44:00Z
1,7,4384,1634,Bill Hilbrich,4,"I cut back on the mayo, and made up the difference with sour cream to adjust the stiffness of the dip.",2001-10-17T16:49:59Z,2001-10-17T16:49:59Z
2,9,4523,2046,Gay Gilmore ckpt,2,i think i did something wrong because i could taste the cornstarch in the finished product.,2000-02-25T09:00:00Z,2000-02-25T09:00:00Z
3,13,7435,1773,Malarkey Test,5,"easily the best i have ever had. juicy flavorful, not dry. the vegetables retain crispness as well, not bland at all!",2000-03-13T21:15:00Z,2000-03-13T21:15:00Z
4,14,44,2085,Tony Small,5,An excellent dish.,2000-03-28T12:51:00Z,2000-03-28T12:51:00Z


In [161]:
#review dataset summary
def dataset_summary(df):
    
    def safe_nunique(col):
        try:
            return col.nunique()
        except TypeError:
            return col.astype(str).nunique()
    
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "missing_count": df.isnull().sum(),
        "missing_pct": (df.isnull().sum() / len(df) * 100).round(2),
        "n_unique": df.apply(safe_nunique)
    }).sort_values(by="missing_count", ascending=False)

    return summary

In [162]:
dataset_summary(recipes)

,dtype,missing_count,missing_pct,n_unique
RecipeYield,object,348071,66.61,34043
AggregatedRating,float64,253223,48.46,9
ReviewCount,float64,247489,47.36,420
RecipeServings,float64,182911,35.01,171
CookTime,object,82545,15.80,490
Keywords,object,17237,3.30,216569
RecipeCategory,object,751,0.14,311
Description,object,5,0.00,492838
RecipeIngredientQuantities,object,3,0.00,459571
Images,object,1,0.00,165889


In [163]:
# fill in missing value function

def fill_na_with_value(df, columns, value=0):
    
    for col in columns:
        if col in df.columns:
            df[col] = df[col].fillna(value)
    
    return df

In [164]:
# Handle missing values in ReviewCount by filling with 0, and create new features based on reviews and ratings
recipes_v1 = fill_na_with_value(recipes, ["ReviewCount"], 0)

recipes_v1["has_reviews"] = recipes_v1["ReviewCount"] > 0
recipes_v1["has_ratings"] = recipes_v1["AggregatedRating"].notna()

recipes_v1["AggregatedRating_na"] = recipes_v1["AggregatedRating"].fillna(0)

recipes_v1["popularity_score"] = (
    recipes_v1["AggregatedRating_na"] * recipes_v1["ReviewCount"]
)


In [165]:
recipes_v1.head()

,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,RecipeCategory,Keywords,RecipeIngredientQuantities,RecipeIngredientParts,AggregatedRating,ReviewCount,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions,has_reviews,has_ratings,AggregatedRating_na,popularity_score
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg"", \n""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/picuaETeN.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/pictzvxW5.jpg"")",Frozen Desserts,"c(""Dessert"", ""Low Protein"", ""Low Cholesterol"", ""Healthy"", ""Free Of..."", ""Summer"", ""Weeknight"", ""Freezer"", ""Easy"")","c(""4"", ""1/4"", ""1"", ""1"")","c(""blueberries"", ""granulated sugar"", ""vanilla yogurt"", ""lemon juice"")",4.5,4.0,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,NaN,"c(""Toss 2 cups berries with sugar."", ""Let stand for 45 minutes, stirring occasionally."", ""Transfer berry-sugar mixture to food processor."", ""Add yogurt and process until smooth."", ""Strain through fine sieve. Pour into baking pan (or transfer to ice cream maker and process according to manufacturers' directions). Freeze uncovered until edges are solid but centre is soft. Transfer to processor and blend until smooth again."", ""Return to pan and freeze until edges are solid."", ""Transfer to processor and blend until smooth again."", \n""Fold in remaining 2 cups of blueberries."", ""Pour into plastic mold and freeze overnight. Let soften slightly to serve."")",True,True,4.5,18.0
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,"c(""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/39/picM9Mhnw.jpg"", ""https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/39/picHv4Ocr.jpg"")",Chicken Breast,"c(""Chicken Thigh & Leg"", ""Chicken"", ""Poultry"", ""Meat"", ""Asian"", ""Indian"", ""Weeknight"", ""Stove Top"")","c(""1"", ""4"", ""2"", ""2"", ""8"", ""1/4"", ""8"", ""1/2"", ""1"", ""1"", ""1/4"", ""1/4"", ""1/2"", ""1/4"", ""2"", ""3"", NA, ""2"", ""1"", ""1"", ""8"", ""2"", ""1/3"", ""1/3"", ""1/3"", ""6"")","c(""saffron"", ""milk"", ""hot green chili peppers"", ""onions"", ""garlic"", ""clove"", ""peppercorns"", ""cardamom seed"", ""cumin seed"", ""poppy seed"", ""mace"", ""cilantro"", ""mint leaf"", ""fresh lemon juice"", ""plain yogurt"", ""boneless chicken"", ""salt"", ""ghee"", ""onion"", ""tomatoes"", ""basmati rice"", ""long-grain rice"", ""raisins"", ""cashews"", ""eggs"")",3.0,1.0,1110.7,58.8,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,NaN,"c(""Soak saffron in warm milk for 5 minutes and puree in blender."", ""Add chiles, onions, ginger, garlic, cloves, peppercorns, cardamom seeds, cinnamon, coriander and cumin seeds, poppy seeds, nutmeg, mace, cilantro or mint leaves and lemon juice. Blend into smooth paste. Put paste into large bowl, add yogurt and mix well."", ""Marinate chicken in yogurt mixture with salt, cove

In [166]:
#reformat columns with c() to list of strings 
def parse_c_vector(x):
    
    if pd.isna(x):
        return []
    
    x = str(x).strip()
    
    if x.startswith("c(") and x.endswith(")"):
        x = x[2:-1]   # remove c( )
        items = re.findall(r'"(.*?)"', x)
        return items
    
    return []

def clean_c_columns(df, columns):

    for col in columns:
        if col in df.columns:
            df[col] = df[col].apply(parse_c_vector)

    return df


In [167]:
c_columns = [
    "Images",
    "Keywords",
    "RecipeIngredientQuantities",
    "RecipeIngredientParts",
    "RecipeInstructions"
]

recipes_v2 = clean_c_columns(recipes_v1, c_columns)

In [168]:
dataset_summary(recipes_v1)

,dtype,missing_count,missing_pct,n_unique
RecipeYield,object,348071,66.61,34043
AggregatedRating,float64,253223,48.46,9
RecipeServings,float64,182911,35.01,171
CookTime,object,82545,15.80,490
RecipeCategory,object,751,0.14,311
Description,object,5,0.00,492838
ProteinContent,float64,0,0.00,2581
SodiumContent,float64,0,0.00,40455
CarbohydrateContent,float64,0,0.00,8102
FiberContent,float64,0,0.00,1067


In [169]:
dataset_summary(recipes_v2)

,dtype,missing_count,missing_pct,n_unique
RecipeYield,object,348071,66.61,34043
AggregatedRating,float64,253223,48.46,9
RecipeServings,float64,182911,35.01,171
CookTime,object,82545,15.80,490
RecipeCategory,object,751,0.14,311
Description,object,5,0.00,492838
ProteinContent,float64,0,0.00,2581
SodiumContent,float64,0,0.00,40455
CarbohydrateContent,float64,0,0.00,8102
FiberContent,float64,0,0.00,1067


In [171]:
recipes_v3 = recipes_v2.copy()

recipes_v3 = recipes_v3.drop(columns=["RecipeYield"])

# AggregatedRating
recipes_v3["AggregatedRating_was_missing"] = recipes_v3["AggregatedRating"].isna()
recipes_v3["AggregatedRating_fill"] = recipes_v3["AggregatedRating"].fillna(recipes_v3["AggregatedRating"].mean())

# RecipeServings
recipes_v3["RecipeServings_was_missing"] = recipes_v3["RecipeServings"].isna()
recipes_v3["RecipeServings_fill"] = recipes_v3["RecipeServings"].fillna(recipes_v3["RecipeServings"].median())

# CookTime_Minutes
#recipes_v3["CookTime_Minutes_was_missing"] = recipes_v3["CookTime_Minutes"].isna()
#recipes_v3["CookTime_Minutes_fill"] = recipes_v3["CookTime_Minutes"].fillna(recipes_v3["CookTime_Minutes"].median())

In [172]:
#convert prepTime and cookTime to minutes
def iso_to_minutes(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x)
    h = re.search(r"(\d+)H", x)
    m = re.search(r"(\d+)M", x)

    hours = int(h.group(1)) if h else 0
    mins = int(m.group(1)) if m else 0

    return hours * 60 + mins

def add_time_minutes(df, columns):
    for col in columns:
        if col in df.columns:
            df[col + "_Minutes"] = df[col].apply(iso_to_minutes)
    return df

def clean_date_column(df, column):
    if column in df.columns:
        df[column + "_Cleaned"] = pd.to_datetime(df[column], errors="coerce")
    return df


In [173]:
recipes_v4 = add_time_minutes(recipes_v3, ["CookTime", "PrepTime", "TotalTime"])

In [174]:
recipes_v4 = clean_date_column(recipes_v4, "DatePublished")

In [175]:
# CookTime_Minutes
recipes_v3["CookTime_Minutes_was_missing"] = recipes_v3["CookTime_Minutes"].isna()
recipes_v3["CookTime_Minutes_fill"] = recipes_v3["CookTime_Minutes"].fillna(recipes_v3["CookTime_Minutes"].median())


In [176]:
# Ingredient normalization

#	1.	lowercase + strip
#	2.	phrase fixes
#	3.	safe descriptor removal
#	4.	safe singularization
##	5.	curated map
#	6.	optional auto-synonym map
#	7.	final manual correction map
#	8.	deduplicate per recipe


class IngredientNormalizer:
    def __init__(
        self,
        descriptor_words=None,
        phrase_fixes=None,
        ingredient_map=None,
        protected_ingredients=None,
        final_fixes=None,
    ):
        self.descriptor_words = descriptor_words or set()
        self.phrase_fixes = phrase_fixes or {}
        self.ingredient_map = ingredient_map or {}
        self.protected_ingredients = protected_ingredients or set()
        self.final_fixes = final_fixes or {}

    #1st basic cleaning
    def normalize_text(self, text):
        if not isinstance(text, str):
            return ""
        text = text.lower().strip()
        text = re.sub(r"[^a-z\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def remove_descriptors(self, text):
        if not text:
            return ""

        for word in sorted(self.descriptor_words, key=len, reverse=True):
            text = re.sub(rf"\b{re.escape(word)}\b", "", text)

        text = re.sub(r"\s+", " ", text).strip()
        return text

    def simple_singularize(self, text):
        if not isinstance(text, str) or not text:
            return ""

        irregular = {
            "tomatoes": "tomato",
            "potatoes": "potato",
            "leaves": "leaf",
            "halves": "half",
            "wolves": "wolf",
            "knives": "knife",
            "wives": "wife",
            "loaves": "loaf",
            "lives": "life",
            "selves": "self",
            "mushrooms": "mushroom",
            "onions": "onion",
            "cloves": "clove",
            "peppers": "pepper",
            "chiles": "chile",
            "beans": "bean",
            "lentils": "lentil",
            "strawberries": "strawberry",
            "blueberries": "blueberry",
            "raspberries": "raspberry",
            "blackberries": "blackberry",
            "cranberries": "cranberry",
            "cherries": "cherry",
        }

        words = text.split()
        if not words:
            return text

        last = words[-1]

        if last in irregular:
            last = irregular[last]
        elif last.endswith("ies") and len(last) > 3:
            last = last[:-3] + "y"
        elif last.endswith("oes") and len(last) > 3:
            last = last[:-2]
        elif last.endswith("s") and not last.endswith("ss") and len(last) > 3:
            last = last[:-1]

        words[-1] = last
        return " ".join(words)

    #main canonicalization
    def canonicalize_ingredient(self, text):
        if not isinstance(text, str):
            return ""

        text = self.normalize_text(text)

        if text in self.phrase_fixes:
            text = self.phrase_fixes[text]

        text = self.remove_descriptors(text)

        if text in self.phrase_fixes:
            text = self.phrase_fixes[text]

        text = self.simple_singularize(text)

        if text in self.ingredient_map:
            text = self.ingredient_map[text]

        if text in self.final_fixes:
            text = self.final_fixes[text]

        text = re.sub(r"\s+", " ", text).strip()
        return text

    def canonicalize_ingredient_list(self, ingredients):
        if not isinstance(ingredients, list):
            return []

        cleaned = []
        for ing in ingredients:
            canon = self.canonicalize_ingredient(ing)
            if canon:
                cleaned.append(canon)

        return list(dict.fromkeys(cleaned))

    # explode ingredients
    def explode_ingredients(self, df, ingredient_col="RecipeIngredientParts", recipe_id_col="RecipeId"):
        rows = []

        for recipe_id, ingredients in zip(df[recipe_id_col], df[ingredient_col]):
            if isinstance(ingredients, list):
                for ing in ingredients:
                    if isinstance(ing, str) and ing.strip():
                        rows.append({
                            recipe_id_col: recipe_id,
                            "raw_ingredient": ing.strip()
                        })

        return pd.DataFrame(rows)

    # helpers for synonym discovery
    def normalize_for_synonyms(self, text):
        text = self.normalize_text(text)

        if text in self.phrase_fixes:
            text = self.phrase_fixes[text]

        text = self.remove_descriptors(text)

        if text in self.phrase_fixes:
            text = self.phrase_fixes[text]

        text = self.simple_singularize(text)

        if text in self.final_fixes:
            text = self.final_fixes[text]

        return text

    def ingredient_frequency_table(self, ingredient_df, col="normalized"):
        freq = ingredient_df[col].value_counts().reset_index()
        freq.columns = [col, "count"]
        return freq

    def get_head_word(self, text):
        if not isinstance(text, str) or not text.strip():
            return ""
        return text.split()[-1]

    def string_similarity(self, a, b):
        return SequenceMatcher(None, a, b).ratio()

    def build_candidate_synonym_pairs(self, freq_table, min_count=20, min_similarity=0.80):
        freq_dict = dict(zip(freq_table["normalized"], freq_table["count"]))
        ingredients = [x for x in freq_table["normalized"] if freq_dict[x] >= min_count]

        candidates = []

        for i in range(len(ingredients)):
            for j in range(i + 1, len(ingredients)):
                a = ingredients[i]
                b = ingredients[j]

                same_head = self.get_head_word(a) == self.get_head_word(b)
                sim = self.string_similarity(a, b)

                if same_head or sim >= min_similarity:
                    candidates.append({
                        "ingredient_a": a,
                        "ingredient_b": b,
                        "count_a": freq_dict[a],
                        "count_b": freq_dict[b],
                        "same_head": same_head,
                        "similarity": round(sim, 3)
                    })

        candidates_df = pd.DataFrame(candidates)

        if not candidates_df.empty:
            candidates_df = candidates_df.sort_values(
                by=["same_head", "similarity", "count_a", "count_b"],
                ascending=[False, False, False, False]
            )

        return candidates_df

    def choose_canonical_by_frequency(self, group, freq_dict):
        group = list(group)
        return max(group, key=lambda x: freq_dict.get(x, 0))

    def build_headword_synonym_map(self, freq_table, min_count=20, max_group_size=10):
        freq_dict = dict(zip(freq_table["normalized"], freq_table["count"]))

        grouped = {}
        for ing in freq_table["normalized"]:
            if freq_dict[ing] >= min_count:
                head = self.get_head_word(ing)
                grouped.setdefault(head, []).append(ing)

        synonym_map = {}

        for head, items in grouped.items():
            items = sorted(items, key=lambda x: freq_dict[x], reverse=True)

            if 1 < len(items) <= max_group_size:
                canonical = self.choose_canonical_by_frequency(items, freq_dict)

                for item in items:
                    if item != canonical:
                        synonym_map[item] = canonical

        return synonym_map

    def filter_synonym_map(self, synonym_map):
        filtered = {}

        for k, v in synonym_map.items():
            if k in self.protected_ingredients or v in self.protected_ingredients:
                continue
            filtered[k] = v

        return filtered

    def apply_synonym_map(self, ingredients, synonym_map):
        if not isinstance(ingredients, list):
            return []

        mapped = []
        for ing in ingredients:
            canon = synonym_map.get(ing, ing)
            if canon:
                mapped.append(canon)

        return list(dict.fromkeys(mapped))

    def apply_final_fixes(self, ingredients):
        if not isinstance(ingredients, list):
            return []

        mapped = [self.final_fixes.get(i, i) for i in ingredients if i]
        return list(dict.fromkeys(mapped))

    #go through full pipeline
    def fit_auto_synonyms(
        self,
        df,
        ingredient_col="RecipeIngredientParts",
        recipe_id_col="RecipeId",
        min_count=30,
        min_similarity=0.80,
        max_group_size=10,
    ):
        ingredient_df = self.explode_ingredients(df, ingredient_col, recipe_id_col)
        ingredient_df["normalized"] = ingredient_df["raw_ingredient"].apply(self.normalize_for_synonyms)

        freq_table = self.ingredient_frequency_table(ingredient_df, "normalized")

        candidate_pairs = self.build_candidate_synonym_pairs(
            freq_table,
            min_count=min_count,
            min_similarity=min_similarity
        )

        auto_synonyms = self.build_headword_synonym_map(
            freq_table,
            min_count=min_count,
            max_group_size=max_group_size
        )

        auto_synonyms_filtered = self.filter_synonym_map(auto_synonyms)

        return {
            "ingredient_df": ingredient_df,
            "freq_table": freq_table,
            "candidate_pairs": candidate_pairs,
            "auto_synonyms": auto_synonyms,
            "auto_synonyms_filtered": auto_synonyms_filtered,
        }

    def transform(self, df, ingredient_col="RecipeIngredientParts"):
        df = df.copy()
        df["ingredients_canonical"] = df[ingredient_col].apply(self.canonicalize_ingredient_list)
        return df

    def transform_with_auto_synonyms(self, df, auto_synonym_map):
        df = df.copy()
        df["ingredients_canonical_auto"] = df["ingredients_canonical"].apply(
            lambda x: self.apply_synonym_map(x, auto_synonym_map)
        )
        df["ingredients_canonical_final"] = df["ingredients_canonical_auto"].apply(
            self.apply_final_fixes
        )
        return df

    # take summaries
    def summarize_ingredients_column(self, df, col="ingredients_canonical_final"):
        total_recipes = len(df)
        ingredient_lists = df[col].dropna()

        total_ingredient_mentions = sum(len(i) for i in ingredient_lists if isinstance(i, list))

        all_ingredients = [
            ing
            for lst in ingredient_lists
            if isinstance(lst, list)
            for ing in lst
        ]

        unique_ingredients = set(all_ingredients)

        print("DATASET SUMMARY")
        print("------------------------")
        print("Total recipes:", total_recipes)
        print("Recipes with ingredient lists:", len(ingredient_lists))
        print("Total ingredient mentions:", total_ingredient_mentions)
        print("Unique ingredients:", len(unique_ingredients))
        if len(ingredient_lists) > 0:
            print("Average ingredients per recipe:",
                  round(total_ingredient_mentions / len(ingredient_lists), 2))

        return all_ingredients

    def ingredient_summary(self, df, col="ingredients_canonical_final", recipe_id_col="RecipeId"):
        ingredient_exploded = df[[recipe_id_col, col]].explode(col)

        summary = (
            ingredient_exploded
            .groupby(col)
            .agg(
                count=(recipe_id_col, "count")
                # recipe_ids=(recipe_id_col, lambda x: list(x))
            )
            .reset_index()
            .sort_values("count", ascending=False)
        )

        return summary

In [177]:
ingredient_map = {
    # eggs
    "hard egg": "egg",
    "jumbo egg": "egg",
    
    #sugar
    "granulated sugar": "sugar",
    "white sugar": "sugar",

    "confectioners sugar": "powdered sugar",
    "confectioners' sugar": "powdered sugar",
    "icing sugar": "powdered sugar",
    "powdered sugar": "powdered sugar",

    "brown sugar": "brown sugar",
    "light brown sugar": "brown sugar",
    "dark brown sugar": "brown sugar",

    # garlic
    "bulb of garlic": "garlic",
    "head of garlic": "garlic",
    "heads of garlic": "garlic",
    "bulbs of garlic": "garlic",
    "bottled garlic": "garlic",
    "garlic cloves": "garlic clove",

    # parsley
    "flat leaf parsley": "parsley",
    "italian parsley": "parsley",
    "flat leaf italian parsley": "parsley",
    "curly leaf parsley": "parsley",

    # vanilla / baking
    "real vanilla": "vanilla",
    "real vanilla extract": "vanilla extract",
    "bicarbonate of soda": "baking soda",

    # basic produce
    "frozen carrot": "carrot",
    "green zucchini": "zucchini",
    "yellow zucchini": "zucchini",
    "english cucumber": "cucumber",
    "seedless cucumber": "cucumber",
    "lebanese cucumber": "cucumber",
    "pickling cucumber": "cucumber",
    "kirby cucumber": "cucumber",
    "english seedless cucumber": "cucumber",
    "seedless european cucumber": "cucumber",
    "hass avocado": "avocado",
    "green cabbage": "cabbage",
    "red cabbage": "cabbage",
    "head of cabbage": "cabbage",
    "white cabbage": "cabbage",
    "purple cabbage": "cabbage",
    "frozen broccoli": "broccoli",
    "head broccoli": "broccoli",
    "head of broccoli": "broccoli",
    "head cauliflower": "cauliflower",
    "frozen cauliflower": "cauliflower",

    # asparagus spelling
    "green asparagu": "asparagus",
    "frozen asparagu": "asparagus",
    "white asparagu": "asparagus",

    # herbs / spices
    "dry oregano": "oregano",
    "powdered cumin": "cumin",
    "dry basil": "basil",
    "leaf basil": "basil",
    "mint leaf": "mint",
    "leaf thyme": "thyme",
    "whole thyme": "thyme",
    "sweet paprika": "paprika",
    "hungarian paprika": "paprika",
    "hot paprika": "paprika",
    "sweet hungarian paprika": "paprika",
    "mild paprika": "paprika",
    "red paprika": "paprika",
    "whole nutmeg": "nutmeg",
    "rubbed sage": "sage",
    "green coriander": "coriander",
    "dry tarragon": "tarragon",
    "dry dill weed": "dill weed",
    "green cardamom": "cardamom",
    "green cardamom pod": "cardamom pod",
    "black cardamom pod": "cardamom pod",
    "whole cardamom pod": "cardamom pod",

    # citrus fixes
    "lemons rind of": "lemon zest",
    "lemon zest of": "lemon zest",
    "rind of lemon": "lemon zest",
    "zest of lemon": "lemon zest",
    "lime rind": "lime zest",
    "rind of lime": "lime zest",
    "zest of lime": "lime zest",
    "orange rind": "orange zest",
    "rind of orange": "orange zest",
    "zest of orange": "orange zest",
    "key lime": "lime",

    # dairy / fats
    "vegan margarine": "margarine",
    "unsalted margarine": "margarine",
    "reduced calorie margarine": "margarine",
    "reduced fat margarine": "margarine",
    "soy margarine": "margarine",
    "low fat margarine": "margarine",
    "regular margarine": "margarine",
    "low fat mayonnaise": "mayonnaise",
    "fat free mayonnaise": "mayonnaise",
    "hellmann s mayonnaise": "mayonnaise",
    "best foods mayonnaise": "mayonnaise",
    "prepared mayonnaise": "mayonnaise",
    "low fat buttermilk": "buttermilk",
    "fat free buttermilk": "buttermilk",
    "dry buttermilk": "buttermilk",
    "low fat creme fraiche": "creme fraiche",
    "low fat ricotta": "ricotta",

    # yogurt / milk alternatives
    "plain yogurt": "yogurt",
    "vanilla flavored soymilk": "soymilk",
    "unsweetened soymilk": "soymilk",
    "vanilla soymilk": "soymilk",
    "low fat soymilk": "soymilk",
    "non fat soymilk": "soymilk",

    # mushrooms
    "button mushroom": "mushroom",
    "cremini mushroom": "mushroom",
    "white mushroom": "mushroom",
    "white button mushroom": "mushroom",
    "whole mushroom": "mushroom",
    "brown button mushroom": "mushroom",

    # nuts / dried fruit
    "whole pecan": "pecan",
    "candied pecan": "pecan",
    "english walnut": "walnut",
    "candied walnut": "walnut",
    "golden raisin": "raisin",
    "seedless raisin": "raisin",
    "white raisin": "raisin",
    "sultana raisin": "raisin",
    "unsalted cashew": "cashew",
    "salted cashew": "cashew",
    "salted peanut": "peanut",
    "unsalted peanut": "peanut",
    "unsalted dry peanut": "peanut",
    "spanish peanut": "peanut",
    "unsalted shelled pistachio": "pistachio",

    # fruit
    "frozen strawberry": "strawberry",
    "frozen unsweetened strawberry": "strawberry",
    "frozen sweetened strawberry": "strawberry",
    "frozen whole strawberry": "strawberry",
    "frozen blueberry": "blueberry",
    "unsweetened frozen blueberry": "blueberry",
    "frozen raspberry": "raspberry",
    "frozen unsweetened raspberry": "raspberry",
    "red raspberry": "raspberry",
    "frozen sweetened raspberry": "raspberry",
    "frozen blackberry": "blackberry",
    "mandarin orange": "orange",
    "navel orange": "orange",
    "seedless orange": "orange",
    "frozen cranberry": "cranberry",
    "sweetened cranberry": "cranberry",
    "bosc pear": "pear",
    "bartlett pear": "pear",
    "anjou pear": "pear",
    "asian pear": "pear",
    "canned pear": "pear",
    "red plum": "plum",
    "purple plum": "plum",
    "pitted date": "date",
    "red grapefruit": "grapefruit",
    "pink grapefruit": "grapefruit",
    "seedless watermelon": "watermelon",

    # cherries
    "bing cherry": "cherry",
    "sour cherry": "cherry",
    "frozen cherry": "cherry",
    "red cherry": "cherry",
    "red maraschino cherry": "maraschino cherry",
    "green maraschino cherry": "maraschino cherry",

    # olives
    "pitted black olive": "black olive",
    "oil cured black olive": "black olive",
    "kalamata olive": "olive",
    "pitted olive": "olive",
    "spanish olive": "olive",
    "nicoise olive": "olive",
    "stuffed green olive": "green olive",

    # meats / seafood
    "whole chicken": "chicken",
    "roasting chicken": "chicken",
    "broiler fryer chicken": "chicken",
    "frying chicken": "chicken",
    "fryer chicken": "chicken",
    "stewing chicken": "chicken",
    "broiler chicken": "chicken",
    "chicken legs thigh": "chicken thigh",
    "jumbo shrimp": "shrimp",
    "frozen shrimp": "shrimp",
    "center cut pork chop": "pork chop",
    "pork loin chop": "pork chop",
    "leg of lamb": "lamb",
    "rack of lamb": "lamb",
    "racks of lamb": "lamb",
    "deli ham": "ham",
    "smoked ham": "ham",
    "country ham": "ham",
    "smoked ham hock": "ham hock",
    "canned tuna": "tuna",
    "albacore tuna": "tuna",
    "solid white tuna": "tuna",
    "chunk tuna": "tuna",
    "smoked salmon": "salmon",
    "pink salmon": "salmon",
    "red salmon": "salmon",
    "canned salmon": "salmon",
    "littleneck clam": "clam",
    "canned clam": "clam",
    "turkey kielbasa": "kielbasa",
    "smoked kielbasa": "kielbasa",
    "polska kielbasa": "kielbasa",
    "polish kielbasa": "kielbasa",
    "bay scallop": "scallop",
    "live lobster": "lobster",
    "smoked trout": "trout",
    "corned beef brisket": "beef brisket",
    "beef round": "beef eye round",
    "beef suet": "suet",

    # grains / starches
    "yellow cornmeal": "cornmeal",
    "white cornmeal": "cornmeal",
    "oat": "oat",
    "quick cooking oat": "oat",
    "old fashioned oat": "oat",
    "quick oat": "oat",
    "steel cut oat": "oat",
    "quaker oat": "oat",
    "porridge oat": "oat",
    "quick cooking oatmeal": "oatmeal",
    "old fashioned oatmeal": "oatmeal",
    "quick oatmeal": "oatmeal",
    "macaroni": "elbow macaroni",
    "corkscrew macaroni": "elbow macaroni",
    "shell macaroni": "elbow macaroni",
    "dry linguine": "linguine",
    "israeli couscou": "couscous",
    "barley": "barley",
    "quick cooking barley": "barley",
    "pot barley": "barley",
    "tapioca": "tapioca",
    "minute tapioca": "tapioca",
    "dry tapioca": "tapioca",
    "pearl tapioca": "tapioca",
    "whole wheat english muffin": "english muffin",
    "bulgur wheat": "bulgar wheat",
    "quick cooking grit": "grit",
    "white hominy": "hominy",
    "yellow hominy": "hominy",
    "hash brown": "hash brown",
    "french fry": "french fry",

    # baking / desserts
    "tomato ketchup": "ketchup",
    "heinz ketchup": "ketchup",
    "no salt added ketchup": "ketchup",
    "vegetable shortening": "shortening",
    "crisco shortening": "shortening",
    "butter flavor shortening": "shortening",
    "solid shortening": "shortening",
    "chocolate graham cracker crumb": "graham cracker crumb",
    "chocolate graham cracker": "graham cracker",
    "honey graham cracker": "graham cracker",
    "cinnamon graham cracker": "graham cracker",
    "low fat graham cracker": "graham cracker",
    "graham cracker crust": "graham cracker pie crust",
    "prepared graham cracker crust": "graham cracker pie crust",
    "reduced fat graham cracker crust": "graham cracker pie crust",
    "graham cracker crumb crust": "graham cracker pie crust",
    "mini marshmallow": "miniature marshmallow",
    "colored miniature marshmallow": "miniature marshmallow",
    "white chocolate baking bar": "chocolate bar",
    "white chocolate curl": "chocolate curl",
    "black treacle": "treacle",
    "agar": "agar agar",
    "yeast cake": "yeast",

    # coconut / fruit products
    "flaked coconut": "coconut",
    "sweetened flaked coconut": "coconut",
    "unsweetened coconut": "coconut",
    "sweetened coconut": "coconut",
    "unsweetened flaked coconut": "coconut",
    "angel flake coconut": "coconut",
    "chunk pineapple": "pineapple",
    "unsweetened pineapple chunk": "pineapple chunk",
    "green mango": "mango",
    "green plantain": "plantain",
    "marinated artichoke": "artichoke",
    "frozen spinach": "spinach",
    "frozen okra": "okra",
    "frozen cut okra": "okra",
    "silken tofu": "tofu",
    "low fat tofu": "tofu",

    # drinks / alcohol
    "white rum": "rum",
    "golden rum": "rum",
    "bacardi rum": "rum",
    "apricot brandy": "brandy",
    "cherry brandy": "brandy",
    "apple brandy": "brandy",
    "peach brandy": "brandy",
    "lager beer": "beer",
    "stout beer": "beer",
    "instant coffee": "coffee",
    "brewed coffee": "coffee",
    "strong black coffee": "coffee",
    "espresso coffee": "coffee",
    "black coffee": "coffee",
    "very strong coffee": "coffee",
    "powdered instant coffee": "coffee",
    "black tea": "tea",
    "instant tea": "tea",
    "hickory liquid smoke": "liquid smoke",
    "dry marsala": "marsala",
    "green creme de menthe": "creme de menthe",
    "white creme de menthe": "creme de menthe",

    # legumes / pantry
    "red lentil": "lentil",
    "brown lentil": "lentil",
    "green lentil": "lentil",
    "dry lentil": "lentil",
    "split red lentil": "lentil",
    "dry green lentil": "lentil",
    "yellow lentil": "lentil",
    "french lentil": "lentil",
    "black lentil": "lentil",
    "unsweetened applesauce": "applesauce",
    "chunky applesauce": "applesauce",
    "canned pumpkin": "pumpkin",
    "solid pack pumpkin": "pumpkin",
    "canned solid pack pumpkin": "pumpkin",
    "libby s canned pumpkin": "pumpkin",
    "whole allspice": "allspice",
    "peppercorn": "black peppercorn",
    "whole black peppercorn": "black peppercorn",
    "low sodium tamari": "tamari",
    "white miso": "miso",
    "red miso": "miso",
    "yellow miso": "miso",
    "liquid honey": "honey",
    "clear honey": "honey",
    "liquid stevia": "stevia",
    "gluten": "vital wheat gluten",
    "wheat gluten": "vital wheat gluten",
    "fine semolina": "semolina",

    # spelling cleanup
    "unsulphured molasse": "molasses",
    "blackstrap molasse": "molasses",
    "pomegranate molasse": "pomegranate molasses",
    "daikon radishe": "radish",
    "red radishe": "radish",
    "daikon radish": "radish",
    "red radish": "radish",

    # special approved
    "chipotle chile in adobo": "chipotle chiles in adobo",
}



In [178]:
APPROVED_INGREDIENT_MAP = {
    "hard egg": "egg",
    "jumbo egg": "egg",

    "bulb of garlic": "garlic",
    "head of garlic": "garlic",
    "heads of garlic": "garlic",
    "bulbs of garlic": "garlic",
    "bottled garlic": "garlic",

    "flat leaf parsley": "parsley",
    "italian parsley": "parsley",
    "flat leaf italian parsley": "parsley",
    "curly leaf parsley": "parsley",

    "real vanilla": "vanilla",
    "real vanilla extract": "vanilla extract",
    "bicarbonate of soda": "baking soda",

    "frozen carrot": "carrot",

    "vegan margarine": "margarine",
    "unsalted margarine": "margarine",
    "reduced calorie margarine": "margarine",
    "reduced fat margarine": "margarine",
    "soy margarine": "margarine",
    "low fat margarine": "margarine",
    "regular margarine": "margarine",

    "powdered cumin": "cumin",

    "liquid honey": "honey",
    "clear honey": "honey",

    "dry oregano": "oregano",

    "dry basil": "basil",
    "leaf basil": "basil",

    "whole nutmeg": "nutmeg",

    "low fat mayonnaise": "mayonnaise",
    "fat free mayonnaise": "mayonnaise",
    "hellmann s mayonnaise": "mayonnaise",
    "best foods mayonnaise": "mayonnaise",
    "prepared mayonnaise": "mayonnaise",

    "leaf thyme": "thyme",
    "whole thyme": "thyme",

    "sweet paprika": "paprika",
    "hungarian paprika": "paprika",
    "hot paprika": "paprika",
    "sweet hungarian paprika": "paprika",
    "mild paprika": "paprika",
    "red paprika": "paprika",

    "button mushroom": "mushroom",
    "cremini mushroom": "mushroom",
    "white mushroom": "mushroom",
    "white button mushroom": "mushroom",
    "whole mushroom": "mushroom",
    "brown button mushroom": "mushroom",

    "whole pecan": "pecan",
    "candied pecan": "pecan",

    "english walnut": "walnut",
    "candied walnut": "walnut",

    "golden raisin": "raisin",
    "seedless raisin": "raisin",
    "white raisin": "raisin",
    "sultana raisin": "raisin",

    "green zucchini": "zucchini",
    "yellow zucchini": "zucchini",

    "jumbo shrimp": "shrimp",
    "frozen shrimp": "shrimp",

    "low fat buttermilk": "buttermilk",
    "fat free buttermilk": "buttermilk",
    "dry buttermilk": "buttermilk",

    "tomato ketchup": "ketchup",
    "heinz ketchup": "ketchup",
    "no salt added ketchup": "ketchup",

    "frozen strawberry": "strawberry",
    "frozen unsweetened strawberry": "strawberry",
    "frozen sweetened strawberry": "strawberry",
    "frozen whole strawberry": "strawberry",

    "whole chicken": "chicken",
    "roasting chicken": "chicken",
    "broiler fryer chicken": "chicken",
    "frying chicken": "chicken",
    "fryer chicken": "chicken",
    "stewing chicken": "chicken",
    "broiler chicken": "chicken",

    "vegetable shortening": "shortening",
    "crisco shortening": "shortening",
    "butter flavor shortening": "shortening",
    "solid shortening": "shortening",

    "english cucumber": "cucumber",
    "seedless cucumber": "cucumber",
    "lebanese cucumber": "cucumber",
    "pickling cucumber": "cucumber",
    "kirby cucumber": "cucumber",
    "english seedless cucumber": "cucumber",
    "seedless european cucumber": "cucumber",

    "hass avocado": "avocado",
    "whole allspice": "allspice",

    "chunk pineapple": "pineapple",

    "chunky salsa": "salsa",
    "mild salsa": "salsa",
    "hot salsa": "salsa",
    "tomatillo salsa": "salsa",
    "chipotle salsa": "salsa",

    "frozen spinach": "spinach",

    "deli ham": "ham",
    "smoked ham": "ham",
    "country ham": "ham",

    "sweetened cranberry": "cranberry",
    "frozen cranberry": "cranberry",

    "green coriander": "coriander",

    "unsulphured molasse": "molasses",
    "blackstrap molasse": "molasses",
    "pomegranate molasse": "pomegranate molasses",

    "rubbed sage": "sage",
    "key lime": "lime",

    "frozen blueberry": "blueberry",
    "unsweetened frozen blueberry": "blueberry",

    "pitted black olive": "black olive",
    "oil cured black olive": "black olive",
    "kalamata olive": "olive",
    "pitted olive": "olive",
    "spanish olive": "olive",
    "nicoise olive": "olive",
    "stuffed green olive": "green olive",

    "white rum": "rum",
    "golden rum": "rum",
    "bacardi rum": "rum",

    "mandarin orange": "orange",
    "navel orange": "orange",
    "seedless orange": "orange",

    "green cabbage": "cabbage",
    "red cabbage": "cabbage",
    "head of cabbage": "cabbage",
    "white cabbage": "cabbage",
    "purple cabbage": "cabbage",

    "frozen broccoli": "broccoli",
    "head broccoli": "broccoli",
    "head of broccoli": "broccoli",

    "flaked coconut": "coconut",
    "sweetened flaked coconut": "coconut",
    "unsweetened coconut": "coconut",
    "sweetened coconut": "coconut",
    "unsweetened flaked coconut": "coconut",
    "angel flake coconut": "coconut",

    "salt pork": "pork",

    "chicken legs thigh": "chicken thigh",

    "yellow cornmeal": "cornmeal",
    "white cornmeal": "cornmeal",

    "oat": "oat",
    "quick cooking oat": "oat",
    "old fashioned oat": "oat",
    "quick oat": "oat",
    "steel cut oat": "oat",
    "quaker oat": "oat",
    "porridge oat": "oat",

    "dry tarragon": "tarragon",

    "lager beer": "beer",
    "stout beer": "beer",

    "frozen raspberry": "raspberry",
    "frozen unsweetened raspberry": "raspberry",
    "red raspberry": "raspberry",
    "frozen sweetened raspberry": "raspberry",

    "japanese eggplant": "eggplant",

    "bittersweet chocolate": "chocolate",
    "white chocolate": "chocolate",
    "baking chocolate": "chocolate",
    "cooking chocolate": "chocolate",

    "green mango": "mango",

    "green asparagu": "asparagus",
    "frozen asparagu": "asparagus",
    "white asparagu": "asparagus",

    "apricot brandy": "brandy",
    "cherry brandy": "brandy",
    "apple brandy": "brandy",
    "peach brandy": "brandy",

    "salted peanut": "peanut",
    "unsalted peanut": "peanut",
    "unsalted dry peanut": "peanut",
    "spanish peanut": "peanut",

    "unsweetened applesauce": "applesauce",
    "chunky applesauce": "applesauce",

    "silken tofu": "tofu",
    "low fat tofu": "tofu",

    "green cardamom": "cardamom",

    "canned pumpkin": "pumpkin",
    "solid pack pumpkin": "pumpkin",
    "canned solid pack pumpkin": "pumpkin",
    "libby s canned pumpkin": "pumpkin",

    "dry dill weed": "dill weed",

    "head cauliflower": "cauliflower",
    "frozen cauliflower": "cauliflower",

    "chocolate graham cracker crumb": "graham cracker crumb",

    "pitted date": "date",

    "macaroni": "elbow macaroni",
    "corkscrew macaroni": "elbow macaroni",
    "shell macaroni": "elbow macaroni",

    "quick cooking oatmeal": "oatmeal",
    "old fashioned oatmeal": "oatmeal",
    "quick oatmeal": "oatmeal",

    "bosc pear": "pear",
    "bartlett pear": "pear",
    "anjou pear": "pear",
    "asian pear": "pear",
    "canned pear": "pear",

    "candied lemon peel": "candied lemon peel",
    "candied citron peel": "candied citron peel",

    "bing cherry": "cherry",
    "sour cherry": "cherry",
    "frozen cherry": "cherry",
    "red cherry": "cherry",
    "red maraschino cherry": "maraschino cherry",
    "green maraschino cherry": "maraschino cherry",

    "unsalted cashew": "cashew",
    "salted cashew": "cashew",

    "vanilla flavored soymilk": "soymilk",
    "unsweetened soymilk": "soymilk",
    "vanilla soymilk": "soymilk",
    "low fat soymilk": "soymilk",
    "non fat soymilk": "soymilk",

    "peppercorn": "black peppercorn",
    "whole black peppercorn": "black peppercorn",

    "leg of lamb": "lamb",
    "rack of lamb": "lamb",
    "racks of lamb": "lamb",

    "unsweetened pineapple chunk": "pineapple chunk",

    "frozen corn kernel": "corn kernel",
    "canned corn kernel": "corn kernel",

    "mini marshmallow": "miniature marshmallow",
    "colored miniature marshmallow": "miniature marshmallow",

    "dry linguine": "linguine",

    "canned tuna": "tuna",
    "albacore tuna": "tuna",
    "solid white tuna": "tuna",
    "chunk tuna": "tuna",

    "hickory liquid smoke": "liquid smoke",

    "black currant": "currant",
    "red currant": "currant",

    "frozen rhubarb": "rhubarb",
    "israeli couscou": "couscous",

    "red lentil": "lentil",
    "brown lentil": "lentil",
    "green lentil": "lentil",
    "dry lentil": "lentil",
    "split red lentil": "lentil",
    "dry green lentil": "lentil",
    "yellow lentil": "lentil",
    "french lentil": "lentil",
    "black lentil": "lentil",

    "low fat creme fraiche": "creme fraiche",

    "chocolate graham cracker": "graham cracker",
    "honey graham cracker": "graham cracker",
    "cinnamon graham cracker": "graham cracker",
    "low fat graham cracker": "graham cracker",

    "red beet": "beet",
    "silver beet": "beet",

    "smoked salmon": "salmon",
    "pink salmon": "salmon",
    "red salmon": "salmon",
    "canned salmon": "salmon",

    "instant coffee": "coffee",
    "brewed coffee": "coffee",
    "strong black coffee": "coffee",
    "espresso coffee": "coffee",
    "black coffee": "coffee",
    "very strong coffee": "coffee",
    "powdered instant coffee": "coffee",

    "frozen blackberry": "blackberry",

    "littleneck clam": "clam",
    "canned clam": "clam",

    "unsalted shelled pistachio": "pistachio",

    "frozen puff pastry": "puff pastry",
    "premade puff pastry": "puff pastry",

    "french baguette": "baguette",
    "sourdough baguette": "baguette",

    "powdered saffron": "saffron",

    "chipotle chile in adobo": "chipotle chiles in adobo",

    "turkey kielbasa": "kielbasa",
    "smoked kielbasa": "kielbasa",
    "polska kielbasa": "kielbasa",
    "polish kielbasa": "kielbasa",

    "center cut pork chop": "pork chop",
    "pork loin chop": "pork chop",

    "low sodium tamari": "tamari",

    "barley": "barley",
    "quick cooking barley": "barley",
    "pot barley": "barley",

    "instant chicken bouillon": "chicken bouillon",
    "chicken flavor instant bouillon": "chicken bouillon",

    "daikon radishe": "radish",
    "red radishe": "radish",
    "daikon radish": "radish",
    "red radish": "radish",

    "bay scallop": "scallop",

    "graham cracker crust": "graham cracker pie crust",
    "prepared graham cracker crust": "graham cracker pie crust",
    "reduced fat graham cracker crust": "graham cracker pie crust",
    "graham cracker crumb crust": "graham cracker pie crust",

    "seedless watermelon": "watermelon",

    "green cardamom pod": "cardamom pod",
    "black cardamom pod": "cardamom pod",
    "whole cardamom pod": "cardamom pod",

    "whole wheat english muffin": "english muffin",
    "corned beef brisket": "beef brisket",

    "frozen okra": "okra",
    "frozen cut okra": "okra",

    "red plum": "plum",
    "purple plum": "plum",

    "hash brown": "hash brown",

    "summer savory": "savory",
    "marinated artichoke": "artichoke",

    "liquid stevia": "stevia",

    "white miso": "miso",
    "red miso": "miso",
    "yellow miso": "miso",

    "pink grapefruit": "grapefruit",
    "red grapefruit": "grapefruit",

    "gluten": "vital wheat gluten",
    "wheat gluten": "vital wheat gluten",

    "fine semolina": "semolina",

    "green onions with top": "green onion",
    "scallion top": "scallion",

    "tapioca": "tapioca",
    "minute tapioca": "tapioca",
    "dry tapioca": "tapioca",
    "pearl tapioca": "tapioca",

    "black tea": "tea",
    "instant tea": "tea",

    "live lobster": "lobster",

    "green plantain": "plantain",

    "white hominy": "hominy",
    "yellow hominy": "hominy",

    "smoked ham hock": "ham hock",
    "white chocolate curl": "chocolate curl",

    "quick cooking grit": "grit",
    "dry marsala": "marsala",
    "bulgur wheat": "bulgar wheat",

    "green creme de menthe": "creme de menthe",
    "white creme de menthe": "creme de menthe",

    "beef suet": "suet",
    "smoked trout": "trout",
    "beef round": "beef eye round",

    "white chocolate baking bar": "chocolate bar",
    "low fat ricotta": "ricotta",

    "french fry": "french fry",
    "yeast cake": "yeast",

    "black treacle": "treacle",
    "agar": "agar agar",
}


In [179]:
CANONICAL_INGREDIENT_MAP = {
    # --------------------
    # onions / alliums
    # --------------------
    "onions": "onion",
    "yellow onion": "onion",
    "yellow onions": "onion",
    "white onion": "onion",
    "white onions": "onion",
    "red onion": "onion",
    "red onions": "onion",
    "sweet onion": "onion",
    "sweet onions": "onion",
    "vidalia onion": "onion",
    "vidalia onions": "onion",

    "green onion": "green onion",
    "green onions": "green onion",
    "spring onion": "green onion",
    "spring onions": "green onion",
    "scallions": "green onion",
    "green onions with top": "green onion",

    "shallots": "shallot",

    "garlic cloves": "garlic",
    "garlic clove": "garlic",
    "bulb of garlic": "garlic",
    "head of garlic": "garlic",
    "heads of garlic": "garlic",
    "bulbs of garlic": "garlic",
    "bottled garlic": "garlic",

    "chives": "chive",

    # --------------------
    # peppers / chiles
    # --------------------
    "bell peppers": "bell pepper",

    "green bell peppers": "green bell pepper",
    "red bell peppers": "red bell pepper",
    "yellow bell peppers": "yellow bell pepper",
    "orange bell peppers": "orange bell pepper",

    # common recipe shorthand
    "green pepper": "green bell pepper",
    "green peppers": "green bell pepper",

    "red pepper": "red bell pepper",
    "red peppers": "red bell pepper",

    # generic chili
    "chili peppers": "chili pepper",
    "chile peppers": "chili pepper",
    "chile pepper": "chili pepper",

    # green chili (keep distinct)
    "green chili peppers": "green chili",
    "green chili pepper": "green chili",
    "green chiles": "green chili",

    "hot green chili peppers": "green chili",
    "hot green chili pepper": "green chili",

    # red chili
    "red chili peppers": "red chili",
    "red chiles": "red chili",

    # specific peppers (important to preserve)
    "jalapenos": "jalapeno",
    "jalapeno peppers": "jalapeno",

    "serrano peppers": "serrano pepper",
    "serrano chiles": "serrano pepper",

    "anaheim peppers": "anaheim pepper",
    "anaheim chiles": "anaheim pepper",

    "poblano peppers": "poblano pepper",
    "poblano chiles": "poblano pepper",

    # chipotle (smoked jalapeno)
    "chipotle peppers": "chipotle chili",
    "chipotle chiles": "chipotle chili",

    # --------------------
    # tomatoes
    # --------------------
    "tomatoes": "tomato",
    "roma tomatoes": "tomato",
    "plum tomatoes": "tomato",
    "cherry tomatoes": "tomato",
    "grape tomatoes": "tomato",
    "canned tomatoes": "tomato",
    "diced tomatoes": "tomato",
    "crushed tomatoes": "tomato",
    "stewed tomatoes": "tomato",

    "tomato sauce": "tomato sauce",
    "tomato paste": "tomato paste",
    "tomato puree": "tomato puree",
    "plain tomato juice": "tomato juice",

    # --------------------
    # potatoes / roots
    # --------------------
    "potatoes": "potato",
    "russet potatoes": "potato",
    "red potatoes": "potato",
    "white potatoes": "potato",
    "new potatoes": "potato",
    "baby potatoes": "potato",

    "sweet potatoes": "sweet potato",
    "yams": "sweet potato",

    "carrots": "carrot",
    "frozen carrot": "carrot",

    "beets": "beet",
    "red beet": "beet",
    "silver beet": "beet",

    "radishes": "radish",
    "daikon radish": "radish",
    "daikon radishe": "radish",
    "red radish": "radish",
    "red radishe": "radish",

    "turnips": "turnip",
    "parsnips": "parsnip",

    # --------------------
    # mushrooms
    # --------------------
    "mushrooms": "mushroom",
    "button mushroom": "mushroom",
    "button mushrooms": "mushroom",
    "white mushroom": "mushroom",
    "white mushrooms": "mushroom",
    "white button mushroom": "mushroom",
    "brown button mushroom": "mushroom",
    "whole mushroom": "mushroom",
    "cremini mushroom": "mushroom",
    "cremini mushrooms": "mushroom",

    # keep special mushroom types separate if you want richer ingredient identity
    "portobello mushrooms": "portobello mushroom",
    "portabella mushrooms": "portobello mushroom",
    "shiitake mushrooms": "shiitake mushroom",
    "porcini mushrooms": "porcini mushroom",

    # --------------------
    # leafy greens / veg
    # --------------------
    "cabbages": "cabbage",
    "green cabbage": "cabbage",
    "red cabbage": "cabbage",
    "white cabbage": "cabbage",
    "purple cabbage": "cabbage",
    "head of cabbage": "cabbage",

    "lettuces": "lettuce",
    "romaine lettuces": "romaine lettuce",

    "spinaches": "spinach",
    "frozen spinach": "spinach",

    "broccoli florets": "broccoli",
    "frozen broccoli": "broccoli",
    "head broccoli": "broccoli",
    "head of broccoli": "broccoli",

    "cauliflower florets": "cauliflower",
    "frozen cauliflower": "cauliflower",
    "head cauliflower": "cauliflower",

    "celery stalks": "celery",
    "celery ribs": "celery",

    "eggplants": "eggplant",
    "japanese eggplant": "eggplant",

    "zucchinis": "zucchini",
    "green zucchini": "zucchini",
    "yellow zucchini": "zucchini",

    "cucumbers": "cucumber",
    "english cucumber": "cucumber",
    "seedless cucumber": "cucumber",
    "lebanese cucumber": "cucumber",
    "pickling cucumber": "cucumber",
    "kirby cucumber": "cucumber",

    # --------------------
    # fruit
    # --------------------
    "lemons": "lemon",
    "limes": "lime",
    "oranges": "orange",
    "mandarin orange": "orange",
    "navel orange": "orange",
    "seedless orange": "orange",

    "apples": "apple",
    "bananas": "banana",
    "pears": "pear",
    "bosc pear": "pear",
    "bartlett pear": "pear",
    "anjou pear": "pear",
    "asian pear": "pear",

    "blueberries": "blueberry",
    "frozen blueberry": "blueberry",
    "unsweetened frozen blueberry": "blueberry",

    "strawberries": "strawberry",
    "frozen strawberry": "strawberry",
    "frozen unsweetened strawberry": "strawberry",
    "frozen sweetened strawberry": "strawberry",
    "frozen whole strawberry": "strawberry",

    "raspberries": "raspberry",
    "frozen raspberry": "raspberry",
    "frozen unsweetened raspberry": "raspberry",
    "frozen sweetened raspberry": "raspberry",
    "red raspberry": "raspberry",

    "blackberries": "blackberry",
    "frozen blackberry": "blackberry",

    "cranberries": "cranberry",
    "frozen cranberry": "cranberry",
    "sweetened cranberry": "cranberry",

    "cherries": "cherry",
    "bing cherry": "cherry",
    "sour cherry": "cherry",
    "red cherry": "cherry",
    "frozen cherry": "cherry",

    "pineapples": "pineapple",
    "chunk pineapple": "pineapple",
    "unsweetened pineapple chunk": "pineapple chunk",

    "mangoes": "mango",
    "green mango": "mango",

    "grapefruits": "grapefruit",
    "pink grapefruit": "grapefruit",
    "red grapefruit": "grapefruit",

    "watermelons": "watermelon",
    "seedless watermelon": "watermelon",

    # --------------------
    # citrus zest / peel
    # --------------------
    "lemons rind of": "lemon zest",
    "lemon zest of": "lemon zest",
    "rind of lemon": "lemon zest",
    "zest of lemon": "lemon zest",
    "lemon rind": "lemon zest",

    "lime rind": "lime zest",
    "rind of lime": "lime zest",
    "zest of lime": "lime zest",
    "lime peel": "lime zest",

    "orange rind": "orange zest",
    "rind of orange": "orange zest",
    "zest of orange": "orange zest",
    "orange peel": "orange zest",

    # --------------------
    # herbs
    # --------------------
    "parsley leaves": "parsley",
    "flat leaf parsley": "parsley",
    "italian parsley": "parsley",
    "flat leaf italian parsley": "parsley",
    "curly leaf parsley": "parsley",

    "basil leaves": "basil",
    "dry basil": "basil",
    "leaf basil": "basil",

    "mint leaves": "mint",
    "mint leaf": "mint",

    "cilantro leaves": "cilantro",
    "coriander leaves": "cilantro",

    "chives": "chive",
    "green coriander": "coriander",

    "leaf thyme": "thyme",
    "whole thyme": "thyme",

    "dry oregano": "oregano",

    "rubbed sage": "sage",
    "dry dill weed": "dill weed",
    "dry tarragon": "tarragon",

    # --------------------
    # spices / seeds
    # --------------------
    "peppercorns": "peppercorn",
    "whole black peppercorn": "black peppercorn",
    "peppercorn": "peppercorn",  # safer than forcing black peppercorn
    "black peppercorns": "black peppercorn",

    "cumin seeds": "cumin seed",
    "cardamom seeds": "cardamom seed",
    "poppy seeds": "poppy seed",
    "mustard seeds": "mustard seed",
    "sesame seeds": "sesame seed",

    "powdered cumin": "cumin",
    "ground cumin": "cumin",
    "whole nutmeg": "nutmeg",

    "sweet paprika": "paprika",
    "hungarian paprika": "paprika",
    "hot paprika": "paprika",
    "sweet hungarian paprika": "paprika",
    "mild paprika": "paprika",
    "red paprika": "paprika",

    "green cardamom": "cardamom",
    "green cardamom pod": "cardamom pod",
    "black cardamom pod": "cardamom pod",
    "whole cardamom pod": "cardamom pod",

    "whole allspice": "allspice",
    "bicarbonate of soda": "baking soda",

    # --------------------
    # dairy / eggs
    # --------------------
    "eggs": "egg",
    "hard egg": "egg",
    "jumbo egg": "egg",

    "whole milk": "milk",
    "skim milk": "milk",
    "low fat milk": "milk",

    "plain yogurt": "yogurt",
    "vanilla yogurt": "vanilla yogurt",

    "low fat buttermilk": "buttermilk",
    "fat free buttermilk": "buttermilk",
    "dry buttermilk": "buttermilk",

    "cream cheese with chive": "cream cheese",
    "low fat creme fraiche": "creme fraiche",
    "low fat ricotta": "ricotta",

    "vegan margarine": "margarine",
    "unsalted margarine": "margarine",
    "reduced calorie margarine": "margarine",
    "reduced fat margarine": "margarine",
    "soy margarine": "margarine",
    "low fat margarine": "margarine",
    "regular margarine": "margarine",

    "low fat mayonnaise": "mayonnaise",
    "fat free mayonnaise": "mayonnaise",
    "hellmann s mayonnaise": "mayonnaise",
    "best foods mayonnaise": "mayonnaise",
    "prepared mayonnaise": "mayonnaise",

    # --------------------
    # oils / sauces / condiments
    # --------------------
    "extra virgin olive oil": "olive oil",
    "olive oils": "olive oil",
    "vegetable oils": "vegetable oil",

    "tomato ketchup": "ketchup",
    "heinz ketchup": "ketchup",
    "no salt added ketchup": "ketchup",

    "soy sauces": "soy sauce",
    "low sodium soy sauce": "soy sauce",
    "low sodium tamari": "tamari",

    "worcestershire sauces": "worcestershire sauce",
    "plain tomato juice": "tomato juice",
    "hickory liquid smoke": "liquid smoke",

    # --------------------
    # grains / starches
    # --------------------
    "yellow cornmeal": "cornmeal",
    "white cornmeal": "cornmeal",

    "oat": "oat",
    "oats": "oat",
    "quick cooking oat": "oat",
    "old fashioned oat": "oat",
    "quick oat": "oat",
    "steel cut oat": "oat",
    "quaker oat": "oat",
    "porridge oat": "oat",

    "quick cooking oatmeal": "oatmeal",
    "old fashioned oatmeal": "oatmeal",
    "quick oatmeal": "oatmeal",

    "macaroni": "elbow macaroni",
    "corkscrew macaroni": "elbow macaroni",
    "shell macaroni": "elbow macaroni",

    "dry linguine": "linguine",
    "barley": "barley",
    "quick cooking barley": "barley",
    "pot barley": "barley",
    "bulgur wheat": "bulgur",
    "israeli couscou": "couscous",

    "tapioca": "tapioca",
    "minute tapioca": "tapioca",
    "dry tapioca": "tapioca",
    "pearl tapioca": "tapioca",

    "graham cracker crumbs": "graham cracker crumb",
    "chocolate graham cracker crumb": "graham cracker crumb",
    "graham cracker crust": "graham cracker pie crust",
    "prepared graham cracker crust": "graham cracker pie crust",
    "reduced fat graham cracker crust": "graham cracker pie crust",
    "graham cracker crumb crust": "graham cracker pie crust",

    # --------------------
    # legumes / nuts
    # --------------------
    "cashews": "cashew",
    "unsalted cashew": "cashew",
    "salted cashew": "cashew",

    "peanuts": "peanut",
    "salted peanut": "peanut",
    "unsalted peanut": "peanut",
    "unsalted dry peanut": "peanut",
    "spanish peanut": "peanut",

    "whole pecan": "pecan",
    "candied pecan": "pecan",
    "pecan halves": "pecan half",
    "pecan halve": "pecan half",

    "english walnut": "walnut",
    "candied walnut": "walnut",
    "walnut halves": "walnut half",
    "walnut halve": "walnut half",

    "golden raisin": "raisin",
    "seedless raisin": "raisin",
    "white raisin": "raisin",
    "sultana raisin": "raisin",

    "red lentil": "lentil",
    "brown lentil": "lentil",
    "green lentil": "lentil",
    "dry lentil": "lentil",
    "split red lentil": "lentil",
    "dry green lentil": "lentil",
    "yellow lentil": "lentil",
    "french lentil": "lentil",
    "black lentil": "lentil",

    "black currant": "currant",
    "red currant": "currant",

    # --------------------
    # meats / seafood
    # --------------------
    "whole chicken": "chicken",
    "roasting chicken": "chicken",
    "broiler fryer chicken": "chicken",
    "frying chicken": "chicken",
    "fryer chicken": "chicken",
    "stewing chicken": "chicken",
    "broiler chicken": "chicken",
    "boneless chicken": "chicken",

    "chicken legs thigh": "chicken thigh",

    "deli ham": "ham",
    "smoked ham": "ham",
    "country ham": "ham",

    "jumbo shrimp": "shrimp",
    "frozen shrimp": "shrimp",

    "center cut pork chop": "pork chop",
    "pork loin chop": "pork chop",
    "salt pork": "pork",

    "leg of lamb": "lamb",
    "rack of lamb": "lamb",
    "racks of lamb": "lamb",

    "canned tuna": "tuna",
    "albacore tuna": "tuna",
    "solid white tuna": "tuna",
    "chunk tuna": "tuna",

    "smoked salmon": "salmon",
    "pink salmon": "salmon",
    "red salmon": "salmon",
    "canned salmon": "salmon",

    "littleneck clam": "clam",
    "canned clam": "clam",
    "bay scallop": "scallop",

    "turkey kielbasa": "kielbasa",
    "smoked kielbasa": "kielbasa",
    "polska kielbasa": "kielbasa",
    "polish kielbasa": "kielbasa",

    "corned beef brisket": "beef brisket",
    "beef round": "beef eye round",
    "beef suet": "suet",
    "smoked trout": "trout",
    "live lobster": "lobster",

    # --------------------
    # pantry / misc
    # --------------------
    "green coriander": "coriander",
    "rubbed sage": "sage",

    "unsulphured molasse": "molasses",
    "blackstrap molasse": "molasses",
    "pomegranate molasse": "pomegranate molasses",

    "green mango": "mango",
    "chunk pineapple": "pineapple",
    "unsweetened pineapple chunk": "pineapple chunk",
    "marinated artichoke": "artichoke",

    "frozen blueberry": "blueberry",
    "unsweetened frozen blueberry": "blueberry",
    "frozen cranberry": "cranberry",
    "sweetened cranberry": "cranberry",
    "frozen rhubarb": "rhubarb",
    "frozen blackberry": "blackberry",

    "canned pumpkin": "pumpkin",
    "solid pack pumpkin": "pumpkin",
    "canned solid pack pumpkin": "pumpkin",
    "libby s canned pumpkin": "pumpkin",

    "frozen corn kernel": "corn kernel",
    "canned corn kernel": "corn kernel",

    "mini marshmallow": "miniature marshmallow",
    "colored miniature marshmallow": "miniature marshmallow",

    "french baguette": "baguette",
    "sourdough baguette": "baguette",
    "frozen puff pastry": "puff pastry",
    "premade puff pastry": "puff pastry",

    "powdered saffron": "saffron",
    "chipotle chile in adobo": "chipotle chiles in adobo",

    "instant chicken bouillon": "chicken bouillon",
    "chicken flavor instant bouillon": "chicken bouillon",

    "white hominy": "hominy",
    "yellow hominy": "hominy",
    "green plantain": "plantain",

    "white miso": "miso",
    "red miso": "miso",
    "yellow miso": "miso",

    "gluten": "vital wheat gluten",
    "wheat gluten": "vital wheat gluten",
    "fine semolina": "semolina",

    "green onions with top": "green onion",
    "scallion top": "scallion",

    "liquid stevia": "stevia",
    "black tea": "tea",
    "instant tea": "tea",
    "instant coffee": "coffee",
    "brewed coffee": "coffee",
    "strong black coffee": "coffee",
    "espresso coffee": "coffee",
    "black coffee": "coffee",
    "very strong coffee": "coffee",
    "powdered instant coffee": "coffee",

    "white chocolate curl": "chocolate curl",
    "white chocolate baking bar": "white chocolate",
    "black treacle": "treacle",
    "agar": "agar agar",
    "yeast cake": "yeast",

    # spelling cleanup
    "green asparagu": "asparagus",
    "frozen asparagu": "asparagus",
    "white asparagu": "asparagus",
    "daikon radishe": "radish",
    "red radishe": "radish",
    "daikon radish": "radish",
    "red radish": "radish",
}

In [180]:
# -----------------------------------
# 1. safer descriptor words
DESCRIPTOR_WORDS = {
    "fresh", "dried", "organic", "raw", "ripe", "pure", "natural",
    "chopped", "diced", "sliced", "minced", "crushed", "shredded",
    "grated", "ground", "peeled", "trimmed", "cubed", "halved",
    "julienned", "thinly", "finely", "roughly",
    "cooked", "uncooked", "boiled", "fried", "roasted", "toasted",
    "baked", "steamed", "grilled",
    "soft", "firm", "smooth", "thick", "thin",
    "boneless", "skinless", "lean",
    "large", "small", "medium", "baby",
    "optional", "divided", "packed"
}

# -----------------------------------
# 2. phrase fixes
# -----------------------------------
PHRASE_FIXES = {
    "lemon zest of": "lemon zest",
    "lemons rind of": "lemon zest",
    "rind of lemon": "lemon zest",
    "zest of lemon": "lemon zest",

    "lime rind": "lime zest",
    "rind of lime": "lime zest",
    "zest of lime": "lime zest",
    "lime peel": "lime zest",

    "orange rind": "orange zest",
    "rind of orange": "orange zest",
    "zest of orange": "orange zest",
    "orange peel": "orange zest",

    "lime juice of": "lime juice",
    "limes juice of": "lime juice",
    "orange juice of": "orange juice",
    "oranges juice of": "orange juice",

    "mint leaf": "mint",
    "bay leave": "bay leaf",
    "basil leave": "basil leaf",
    "thyme leave": "thyme leaf",
    "parsley leave": "parsley leaf",

    "green chily": "green chili",
    "red chily": "red chili",
    "chily": "chili",
    "anaheim chily": "anaheim chili",
    "serrano chily": "serrano chili",

    "radishe": "radish",
    "asparagu": "asparagus",
    "molasse": "molasses",
}

# -----------------------------------
# 3. final fixes after all mapping
# -----------------------------------
FINAL_FIXES = {
    "lemons rind of": "lemon zest",
    "lemon zest of": "lemon zest",
    "rind of lemon": "lemon zest",
    "zest of lemon": "lemon zest",
    "lime rind": "lime zest",
    "rind of lime": "lime zest",
    "zest of lime": "lime zest",
    "orange rind": "orange zest",
    "rind of orange": "orange zest",
    "zest of orange": "orange zest",
    "mint leaf": "mint",
}

In [181]:
combined_ingredient_map = ingredient_map.copy()
combined_ingredient_map.update(APPROVED_INGREDIENT_MAP)
combined_ingredient_map.update(CANONICAL_INGREDIENT_MAP)

normalizer = IngredientNormalizer(
    descriptor_words=DESCRIPTOR_WORDS,
    phrase_fixes=PHRASE_FIXES,
    ingredient_map=combined_ingredient_map,
    protected_ingredients=PROTECTED_INGREDIENTS,
    final_fixes=FINAL_FIXES
)

# Step 1: canonicalize from raw ingredient lists
recipes_v4["RecipeIngredientParts_old"] = recipes_v4["RecipeIngredientParts"]
recipes_v4 = normalizer.transform(recipes_v4, ingredient_col="RecipeIngredientParts")

# Step 2: fit auto synonym discovery
auto_results = normalizer.fit_auto_synonyms(
    recipes_v3,
    ingredient_col="RecipeIngredientParts",
    recipe_id_col="RecipeId",
    min_count=30,
    min_similarity=0.80,
    max_group_size=10
)

ingredient_df = auto_results["ingredient_df"]
freq_table = auto_results["freq_table"]
candidate_pairs = auto_results["candidate_pairs"]
auto_synonyms = auto_results["auto_synonyms"]
auto_synonyms_filtered = auto_results["auto_synonyms_filtered"]

# Step 3: apply auto synonyms + final fixes
recipes_v5 = normalizer.transform_with_auto_synonyms(
    recipes_v4,
    auto_synonym_map=auto_synonyms_filtered
)


In [182]:
recipes_v4.head()

,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,RecipeCategory,Keywords,RecipeIngredientQuantities,RecipeIngredientParts,AggregatedRating,ReviewCount,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeInstructions,has_reviews,has_ratings,AggregatedRating_na,popularity_score,AggregatedRating_was_missing,AggregatedRating_fill,RecipeServings_was_missing,RecipeServings_fill,CookTime_Minutes,PrepTime_Minutes,TotalTime_Minutes,DatePublished_Cleaned,CookTime_Minutes_was_missing,CookTime_Minutes_fill,RecipeIngredientParts_old,ingredients_canonical
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.,"[https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/picuaETeN.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/pictzvxW5.jpg]",Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Healthy, Free Of..., Summer, Weeknight, Freezer, Easy]","[4, 1/4, 1, 1]","[blueberries, granulated sugar, vanilla yogurt, lemon juice]",4.5,4.0,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,"[Toss 2 cups berries with sugar., Let stand for 45 minutes, stirring occasionally., Transfer berry-sugar mixture to food processor., Add yogurt and process until smooth., Strain through fine sieve. Pour into baking pan (or transfer to ice cream maker and process according to manufacturers' directions). Freeze uncovered until edges are solid but centre is soft. Transfer to processor and blend until smooth again., Return to pan and freeze until edges are solid., Transfer to processor and blend until smooth again., Fold in remaining 2 cups of blueberries., Pour into plastic mold and freeze overnight. Let soften slightly to serve.]",True,True,4.5,18.0,False,4.5,False,4.0,1440.0,45,1485,1999-08-09 21:46:00+00:00,False,1440.0,"[blueberries, granulated sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]"
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,"[https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/39/picM9Mhnw.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/39/picHv4Ocr.jpg]",Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, Asian, Indian, Weeknight, Stove Top]","[1, 4, 2, 2, 8, 1/4, 8, 1/2, 1, 1, 1/4, 1/4, 1/2, 1/4, 2, 3, 2, 1, 1, 8, 2, 1/3, 1/3, 1/3, 6]","[saffron, milk, hot green chili peppers, onions, garlic, clove, peppercorns, cardamom seed, cumin seed, poppy seed, mace, cilantro, mint leaf, fresh lemon juice, plain yogurt, boneless chicken, salt, ghee, onion, tomatoes, basmati rice, long-grain rice, raisins, cashews, eggs]",3.0,1.0,1110.7,58.8,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,"[Soak saffron in warm milk for 5 minutes and puree in blender., Add chiles, onions, ginger, garlic, cloves, peppercorns, cardamom seeds, cinnamon, coriander and cumin seeds, poppy seeds, nutmeg, mace, cilantro or mint leaves and lemon juice. Blend into smooth paste. Put paste into large bowl, add yogurt and mix well., Marinate

Lossy Counting Module

In [183]:
import pandas as pd
from collections import Counter


class LossyIngredientReducer:
    """
    Reduce sparse ingredient vocabularies by removing or replacing
    ingredients based on recipe-level document frequency.

    match_mode:
        'lt' -> frequency < threshold
        'le' -> frequency <= threshold
        'eq' -> frequency == threshold
        'ge' -> frequency >= threshold
    """

    def __init__(self, col="ingredients_canonical_final"):
        self.col = col
        self.docfreq = None
        self.target_set = set()
        self.threshold = None
        self.match_mode = None

    # --------------------------------------------------
    # internal helpers
    # --------------------------------------------------
    @staticmethod
    def _is_valid_list(x):
        return isinstance(x, list)

    @staticmethod
    def _deduplicate_keep_order(items):
        return list(dict.fromkeys(items))

    @staticmethod
    def _safe_len_list(x):
        return len(x) if isinstance(x, list) else 0

    # --------------------------------------------------
    # dataset summary
    # --------------------------------------------------
    def dataset_level_summary(self, df, col=None):
        """
        Summarize a dataset column containing ingredient lists.
        """
        if col is None:
            col = self.col

        total_recipes = len(df)
        valid_lists = df[col].apply(self._is_valid_list)
        ingredient_lists = df.loc[valid_lists, col]

        total_mentions = sum(len(lst) for lst in ingredient_lists)
        all_ingredients = [ing for lst in ingredient_lists for ing in lst]
        unique_ingredients = len(set(all_ingredients))
        avg_per_recipe = round(total_mentions / len(ingredient_lists), 2) if len(ingredient_lists) > 0 else 0.0

        return {
            "column": col,
            "total_recipes": total_recipes,
            "recipes_with_lists": int(valid_lists.sum()),
            "total_mentions": total_mentions,
            "unique_ingredients": unique_ingredients,
            "avg_ingredients_per_recipe": avg_per_recipe
        }

    def dataset_level_summary_df(self, df, col=None):
        return pd.DataFrame([self.dataset_level_summary(df, col=col)])

    # --------------------------------------------------
    # fit
    # --------------------------------------------------
    def fit(self, df, threshold=5, match_mode="lt", col=None):
        if col is not None:
            self.col = col

        self.threshold = threshold
        self.match_mode = match_mode

        counter = Counter()
        for lst in df[self.col]:
            if isinstance(lst, list):
                counter.update(set(lst))   # document frequency

        self.docfreq = (
            pd.DataFrame(counter.items(), columns=["ingredient", "recipe_count"])
            .sort_values(["recipe_count", "ingredient"], ascending=[False, True])
            .reset_index(drop=True)
        )

        if match_mode == "lt":
            mask = self.docfreq["recipe_count"] < threshold
        elif match_mode == "le":
            mask = self.docfreq["recipe_count"] <= threshold
        elif match_mode == "eq":
            mask = self.docfreq["recipe_count"] == threshold
        elif match_mode == "ge":
            mask = self.docfreq["recipe_count"] >= threshold
        else:
            raise ValueError("match_mode must be one of: 'lt', 'le', 'eq', 'ge'")

        self.target_set = set(self.docfreq.loc[mask, "ingredient"])
        return self

    # --------------------------------------------------
    # transform one list
    # --------------------------------------------------
    def transform_list(
        self,
        ingredients,
        mode="remove",
        replacement_token="other_target_ingredient",
        min_remaining=1,
        keep_original_if_empty=True
    ):
        if not isinstance(ingredients, list):
            return []

        original = self._deduplicate_keep_order(ingredients)

        if mode == "remove":
            cleaned = [x for x in original if x not in self.target_set]
            cleaned = self._deduplicate_keep_order(cleaned)

            if len(cleaned) < min_remaining:
                return original if keep_original_if_empty else cleaned

            return cleaned

        elif mode == "replace":
            cleaned = [
                replacement_token if x in self.target_set else x
                for x in original
            ]
            cleaned = self._deduplicate_keep_order(cleaned)

            if len(cleaned) < min_remaining:
                return original if keep_original_if_empty else cleaned

            return cleaned

        else:
            raise ValueError("mode must be 'remove' or 'replace'")

    # --------------------------------------------------
    # transform dataframe
    # --------------------------------------------------
    def transform(
        self,
        df,
        mode="remove",
        replacement_token="other_target_ingredient",
        min_remaining=1,
        keep_original_if_empty=True,
        new_col=None
    ):
        if self.docfreq is None:
            raise ValueError("You must call fit() before transform().")

        df_out = df.copy()

        if new_col is None:
            suffix = "removed" if mode == "remove" else "replaced"
            op = f"{self.match_mode}_{self.threshold}" if self.threshold is not None else "lossy"
            new_col = f"{self.col}_{op}_{suffix}"

        df_out[new_col] = df_out[self.col].apply(
            lambda x: self.transform_list(
                x,
                mode=mode,
                replacement_token=replacement_token,
                min_remaining=min_remaining,
                keep_original_if_empty=keep_original_if_empty
            )
        )

        return df_out

    # --------------------------------------------------
    # convenience fit+transform
    # --------------------------------------------------
    def fit_transform(
        self,
        df,
        threshold=5,
        match_mode="lt",
        mode="remove",
        replacement_token="other_target_ingredient",
        min_remaining=1,
        keep_original_if_empty=True,
        new_col=None,
        col=None
    ):
        self.fit(df, threshold=threshold, match_mode=match_mode, col=col)
        return self.transform(
            df,
            mode=mode,
            replacement_token=replacement_token,
            min_remaining=min_remaining,
            keep_original_if_empty=keep_original_if_empty,
            new_col=new_col
        )

    # --------------------------------------------------
    # compare before / after
    # --------------------------------------------------
    def compare_dataset_levels(self, df, new_col):
        before = self.dataset_level_summary(df, self.col)
        after = self.dataset_level_summary(df, new_col)

        rows = []
        metrics = [
            "total_recipes",
            "recipes_with_lists",
            "total_mentions",
            "unique_ingredients",
            "avg_ingredients_per_recipe"
        ]

        for metric in metrics:
            before_val = before[metric]
            after_val = after[metric]

            pct_change = None
            if isinstance(before_val, (int, float)) and before_val != 0:
                pct_change = round((after_val - before_val) / before_val, 4)

            rows.append({
                "metric": metric,
                "before": before_val,
                "after": after_val,
                "pct_change": pct_change
            })

        return pd.DataFrame(rows)

    # --------------------------------------------------
    # threshold evaluation
    # --------------------------------------------------
    def evaluate_thresholds(
        self,
        df,
        thresholds=(2, 3, 5, 7, 10),
        match_mode="le",
        mode="remove",
        replacement_token="other_target_ingredient",
        min_remaining=1,
        keep_original_if_empty=True
    ):
        """
        Evaluate multiple thresholds with recipe-level impact.
        """
        results = []

        base_summary = self.dataset_level_summary(df, self.col)
        base_unique = base_summary["unique_ingredients"]
        total_recipes_before = base_summary["total_recipes"]

        for t in thresholds:
            reducer = LossyIngredientReducer(col=self.col)
            reducer.fit(df, threshold=t, match_mode=match_mode)

            temp_col = "__temp_lossy__"
            temp_df = reducer.transform(
                df,
                mode=mode,
                replacement_token=replacement_token,
                min_remaining=min_remaining,
                keep_original_if_empty=keep_original_if_empty,
                new_col=temp_col
            )

            temp_summary = reducer.dataset_level_summary(temp_df, temp_col)

            changed_mask = temp_df[self.col] != temp_df[temp_col]
            recipes_changed = int(changed_mask.sum())

            results.append({
                "threshold": t,
                "match_mode": match_mode,
                "target_count": len(reducer.target_set),

                "total_recipes_before": total_recipes_before,
                "total_recipes_after": temp_summary["total_recipes"],

                "remaining_unique": temp_summary["unique_ingredients"],
                "unique_reduction_pct": round(
                    (base_unique - temp_summary["unique_ingredients"]) / base_unique, 4
                ) if base_unique else 0,

                "avg_ingredients_per_recipe_before": base_summary["avg_ingredients_per_recipe"],
                "avg_ingredients_per_recipe_after": temp_summary["avg_ingredients_per_recipe"],

                "recipes_changed_count": recipes_changed,
                "recipes_changed_pct": round(recipes_changed / total_recipes_before, 4) if total_recipes_before else 0
            })

        return pd.DataFrame(results)

    # --------------------------------------------------
    # target / kept ingredients
    # --------------------------------------------------
    def get_target_ingredients_df(self):
        if self.docfreq is None:
            raise ValueError("You must call fit() first.")

        return (
            self.docfreq[self.docfreq["ingredient"].isin(self.target_set)]
            .sort_values(["recipe_count", "ingredient"], ascending=[True, True])
            .reset_index(drop=True)
        )

    def get_kept_ingredients_df(self):
        if self.docfreq is None:
            raise ValueError("You must call fit() first.")

        return (
            self.docfreq[~self.docfreq["ingredient"].isin(self.target_set)]
            .sort_values(["recipe_count", "ingredient"], ascending=[False, True])
            .reset_index(drop=True)
        )

In [185]:
def apply_multiple_thresholds(df, col, thresholds, mode="replace"):
    df_out = df.copy()
    reducer = LossyIngredientReducer(col=col)

    for t in thresholds:
        reducer.fit(df_out, threshold=t, match_mode="le")
        new_col = f"{col}_le_{t}_{mode}"
        
        df_out = reducer.transform(
            df_out,
            mode=mode,
            new_col=new_col
        )

    return df_out


recipes_v5 = apply_multiple_thresholds(
    recipes_v5,
    col="ingredients_canonical_final",
    thresholds=[5],
    mode="replace"
)

In [186]:
reducer = LossyIngredientReducer(col="ingredients_canonical_final")

threshold_results = reducer.evaluate_thresholds(
    recipes_v5,
    thresholds=[0, 1, 2, 3, 4, 5, 7, 10],
    match_mode="le",
    mode="replace"
)

threshold_results.to_csv("threshold_evaluation_results.csv", index=False)

In [187]:
recipes_v5.head()

,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,RecipeCategory,Keywords,RecipeIngredientQuantities,RecipeIngredientParts,AggregatedRating,ReviewCount,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeInstructions,has_reviews,has_ratings,AggregatedRating_na,popularity_score,AggregatedRating_was_missing,AggregatedRating_fill,RecipeServings_was_missing,RecipeServings_fill,CookTime_Minutes,PrepTime_Minutes,TotalTime_Minutes,DatePublished_Cleaned,CookTime_Minutes_was_missing,CookTime_Minutes_fill,RecipeIngredientParts_old,ingredients_canonical,ingredients_canonical_auto,ingredients_canonical_final,ingredients_canonical_final_le_5_replace
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.,"[https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/picuaETeN.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/pictzvxW5.jpg]",Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Healthy, Free Of..., Summer, Weeknight, Freezer, Easy]","[4, 1/4, 1, 1]","[blueberries, granulated sugar, vanilla yogurt, lemon juice]",4.5,4.0,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,"[Toss 2 cups berries with sugar., Let stand for 45 minutes, stirring occasionally., Transfer berry-sugar mixture to food processor., Add yogurt and process until smooth., Strain through fine sieve. Pour into baking pan (or transfer to ice cream maker and process according to manufacturers' directions). Freeze uncovered until edges are solid but centre is soft. Transfer to processor and blend until smooth again., Return to pan and freeze until edges are solid., Transfer to processor and blend until smooth again., Fold in remaining 2 cups of blueberries., Pour into plastic mold and freeze overnight. Let soften slightly to serve.]",True,True,4.5,18.0,False,4.5,False,4.0,1440.0,45,1485,1999-08-09 21:46:00+00:00,False,1440.0,"[blueberries, granulated sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]"
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,"[https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/39/picM9Mhnw.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/39/picHv4Ocr.jpg]",Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, Asian, Indian, Weeknight, Stove Top]","[1, 4, 2, 2, 8, 1/4, 8, 1/2, 1, 1, 1/4, 1/4, 1/2, 1/4, 2, 3, 2, 1, 1, 8, 2, 1/3, 1/3, 1/3, 6]","[saffron, milk, hot green chili peppers, onions, garlic, clove, peppercorns, cardamom seed, cumin seed, poppy seed, mace, cilantro, mint leaf, fresh lemon juice, plain yogurt, boneless chicken, salt, ghee, onion, tomatoes, basmati rice, long-grain rice, raisins, cashews, eggs]",3.0,1.0,1110.7,58.8,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,"[Soak saffron in warm milk for 5 minutes and puree in blender., Add chiles, onions

In [188]:
#WHO standard 

def add_energy_and_who_flags(
    df: pd.DataFrame,
    calories_col="Calories",
    fat_col="FatContent",
    satfat_col="SaturatedFatContent",
    carbs_col="CarbohydrateContent",
    protein_col="ProteinContent",
    sodium_col="SodiumContent",
    who_sodium_mg_limit=667  # per-serving proxy instead of full-day 2000 mg 2000mg/3
) -> pd.DataFrame:
    """
    Adds macro-energy columns, % energy from fat/sat fat,
    WHO compliance flags, and a WHO-based score.
    Assumes nutrient values are per serving.
    """
    df = df.copy()

    needed = [calories_col, fat_col, satfat_col, carbs_col, protein_col, sodium_col]
    for col in needed:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Energy from macros (kcal)
    df["EnergyFromFat"] = df[fat_col] * 9
    df["EnergyFromSaturatedFat"] = df[satfat_col] * 9
    df["EnergyFromCarbs"] = df[carbs_col] * 4
    df["EnergyFromProtein"] = df[protein_col] * 4

    # Avoid divide-by-zero
    denom = df[calories_col].replace({0: np.nan})

    df["SaturatedFatPercentEnergy"] = df["EnergyFromSaturatedFat"] / denom * 100
    df["FatPercentEnergy"] = df["EnergyFromFat"] / denom * 100

    # WHO-style compliance flags
    df["WHO_SatFat_Compliant"] = df["SaturatedFatPercentEnergy"] < 10
    df["WHO_Fat_Compliant"] = df["FatPercentEnergy"] < 30
    df["WHO_Sodium_Compliant"] = df[sodium_col] < who_sodium_mg_limit

    # WHO score: 0 to 3
    flags = ["WHO_SatFat_Compliant", "WHO_Fat_Compliant", "WHO_Sodium_Compliant"]
    df["WHO_Score"] = df[flags].fillna(False).astype(int).sum(axis=1)

    # Optional label
    df["WHO_Healthy"] = df["WHO_Score"] >= 2

    return df


#UK FSA standard
def add_fsa_score(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col in [
        "Calories", "SugarContent", "SaturatedFatContent",
        "SodiumContent", "FiberContent", "ProteinContent"
    ]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["Energy_kJ"] = df["Calories"] * 4.184

    def fsa_bins(x, bins):
        if pd.isna(x):
            return np.nan
        return sum(x > b for b in bins)

    df["A_score"] = (
        df["Energy_kJ"].apply(lambda x: fsa_bins(x, [335,670,1005,1340,1675,2010,2345,2680,3015,3350])) +
        df["SugarContent"].apply(lambda x: fsa_bins(x, [4.5,9,13.5,18,22.5,27,31,36,40,45])) +
        df["SaturatedFatContent"].apply(lambda x: fsa_bins(x, [1,2,3,4,5,6,7,8,9,10])) +
        df["SodiumContent"].apply(lambda x: fsa_bins(x, [90,180,270,360,450,540,630,720,810,900]))
    )

    df["C_score"] = (
        df["FiberContent"].apply(lambda x: fsa_bins(x, [0.9,1.9,2.8,3.7,4.7])) +
        df["ProteinContent"].apply(lambda x: fsa_bins(x, [1.6,3.2,4.8,6.4,8.0]))
    )

    df["FSA_Score"] = df["A_score"] - df["C_score"]
    df["FSA_Healthy"] = df["FSA_Score"] < 4

    return df

In [189]:
recipes_v6 = add_energy_and_who_flags(recipes_v5)
recipes_v6 = add_fsa_score(recipes_v6)

In [190]:
recipes_v6.head()

,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,RecipeCategory,Keywords,RecipeIngredientQuantities,RecipeIngredientParts,AggregatedRating,ReviewCount,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeInstructions,has_reviews,has_ratings,AggregatedRating_na,popularity_score,AggregatedRating_was_missing,AggregatedRating_fill,RecipeServings_was_missing,RecipeServings_fill,CookTime_Minutes,PrepTime_Minutes,TotalTime_Minutes,DatePublished_Cleaned,CookTime_Minutes_was_missing,CookTime_Minutes_fill,RecipeIngredientParts_old,ingredients_canonical,ingredients_canonical_auto,ingredients_canonical_final,ingredients_canonical_final_le_5_replace,EnergyFromFat,EnergyFromSaturatedFat,EnergyFromCarbs,EnergyFromProtein,SaturatedFatPercentEnergy,FatPercentEnergy,WHO_SatFat_Compliant,WHO_Fat_Compliant,WHO_Sodium_Compliant,WHO_Score,WHO_Healthy,Energy_kJ,A_score,C_score,FSA_Score,FSA_Healthy
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen Dessert recipe from Food.com.,"[https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/YUeirxMLQaeE1h3v3qnM_229%20berry%20blue%20frzn%20dess.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/AFPDDHATWzQ0b1CDpDAT_255%20berry%20blue%20frzn%20dess.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/UYgf9nwMT2SGGJCuzILO_228%20berry%20blue%20frzn%20dess.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/PeBMJN2TGSaYks2759BA_20140722_202142.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/picuaETeN.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/38/pictzvxW5.jpg]",Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Healthy, Free Of..., Summer, Weeknight, Freezer, Easy]","[4, 1/4, 1, 1]","[blueberries, granulated sugar, vanilla yogurt, lemon juice]",4.5,4.0,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,"[Toss 2 cups berries with sugar., Let stand for 45 minutes, stirring occasionally., Transfer berry-sugar mixture to food processor., Add yogurt and process until smooth., Strain through fine sieve. Pour into baking pan (or transfer to ice cream maker and process according to manufacturers' directions). Freeze uncovered until edges are solid but centre is soft. Transfer to processor and blend until smooth again., Return to pan and freeze until edges are solid., Transfer to processor and blend until smooth again., Fold in remaining 2 cups of blueberries., Pour into plastic mold and freeze overnight. Let soften slightly to serve.]",True,True,4.5,18.0,False,4.5,False,4.0,1440.0,45,1485,1999-08-09 21:46:00+00:00,False,1440.0,"[blueberries, granulated sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]",22.5,11.7,148.4,12.8,6.846109,13.165594,True,True,True,3,True,715.0456,9,4,5,False
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,"[https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/39/picM9Mhnw.jpg, https://img.sndimg.com/food/image/upload/w_555,h_416,c_fit,fl_progressive,q_95/v1/img/recipes/39/picHv4Ocr.jpg]",Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, Asian, Indian, Weeknight, Stove Top]","[1, 4, 2, 2, 8, 1/4, 8, 1/2, 1, 1, 1/4, 1/4, 1/2, 1/4, 2, 3, 2, 1, 1, 8, 2, 1/3, 1/3, 1/3, 6]","[saffron, milk, hot green chili peppers, onions, garlic, clove, peppercorns, cardamom seed, cum

In [206]:
#Normalization on number columns for better model input

#	Calories
#	FatContent
#	SaturatedFatContent
#	SodiumContent
#	SugarContent
#	FiberContent
#	ProteinContent
#	CarbohydrateContent
#	WHO_Score
#	FSA_Score

#standard scaler
from sklearn.preprocessing import StandardScaler


recipes_v7 = recipes_v6.copy()
recipes_v7["ReviewCount_log"] = np.log1p(recipes_v7["ReviewCount"])

#weighted rating
C = recipes_v7["AggregatedRating"].mean()        # global average rating
m = recipes_v7["ReviewCount"].quantile(0.75)     # threshold

recipes_v7["weighted_rating"] = (
    (recipes_v7["ReviewCount"] / (recipes_v7["ReviewCount"] + m)) * recipes_v7["AggregatedRating_fill"] +
    (m / (recipes_v7["ReviewCount"] + m)) * C
)


recipes_v7["FatPercentEnergy"].fillna(recipes_v7["FatPercentEnergy"].median(), inplace=True)
recipes_v7["SaturatedFatPercentEnergy"].fillna(recipes_v7["SaturatedFatPercentEnergy"].median(), inplace=True)

recipes_v7["RecipeCategory"] = recipes_v7["RecipeCategory"].fillna("N/A")
recipes_v7["Description"] = recipes_v7["Description"].fillna("N/A")


num_cols = [
    # Nutrition
    "Calories",
    "FatContent",
    "SaturatedFatContent",
    "SodiumContent",
    "SugarContent",
    "FiberContent",
    "ProteinContent",
    "CarbohydrateContent",

    # Energy
    "Energy_kJ",
    "EnergyFromFat",
    "EnergyFromSaturatedFat",
    "EnergyFromCarbs",
    "EnergyFromProtein",
    "FatPercentEnergy",
    "SaturatedFatPercentEnergy",

    # Time & structure
    "PrepTime_Minutes",
    "TotalTime_Minutes",
    "CookTime_Minutes_fill",
    "RecipeServings_fill",

    # Engagement
    "AggregatedRating_fill",
    "ReviewCount",
    "weighted_rating",
    "popularity_score", #feature use 

    # Scores
    "WHO_Score",
    "FSA_Score",
    "A_score",
    "C_score"
]


scaler = StandardScaler()

scaled_values = scaler.fit_transform(recipes_v7[num_cols])

# Create new column names
scaled_cols = [col + "_scaled" for col in num_cols]

# Assign back to dataframe
recipes_v7[scaled_cols] = scaled_values


/var/folders/h7/rdkv64ss4rj67v0yrckxlcdc0000gn/T/ipykernel_31801/2983399761.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  recipes_v7["FatPercentEnergy"].fillna(recipes_v7["FatPercentEnergy"].median(), inplace=True)
/var/folders/h7/rdkv64ss4rj67v0yrckxlcdc0000gn/T/ipykernel_31801/2983399761.py:32: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermedi

In [ ]:
recipes_v7.head()

,RecipeId,Name,AuthorId,AuthorName,CookTime,PrepTime,TotalTime,DatePublished,Description,Images,RecipeCategory,Keywords,RecipeIngredientQuantities,RecipeIngredientParts,AggregatedRating,ReviewCount,Calories,FatContent,SaturatedFatContent,CholesterolContent,SodiumContent,CarbohydrateContent,FiberContent,SugarContent,ProteinContent,RecipeServings,RecipeYield,RecipeInstructions,has_reviews,has_ratings,AggregatedRating_na,popularity_score,CookTime_Minutes,PrepTime_Minutes,TotalTime_Minutes,DatePublished_Cleaned,RecipeIngredientParts_old,ingredients_canonical,ingredients_canonical_auto,ingredients_canonical_final,ingredients_canonical_pruned,ingredients_canonical_lossy,ingredients_lossy,ingredients_le_0_replaced,ingredients_le_1_replaced,ingredients_le_2_replaced,ingredients_le_3_replaced,ingredients_le_4_replaced,ingredients_le_5_replaced,ingredients_canonical_final_le_0_replace,ingredients_canonical_final_le_1_replace,ingredients_canonical_final_le_2_replace,ingredients_canonical_final_le_3_replace,ingredients_canonical_final_le_4_replace,ingredients_canonical_final_le_5_replace,EnergyFromFat,EnergyFromSaturatedFat,EnergyFromCarbs,EnergyFromProtein,SaturatedFatPercentEnergy,FatPercentEnergy,WHO_SatFat_Compliant,WHO_Fat_Compliant,WHO_Sodium_Compliant,WHO_Score,WHO_Healthy,Energy_kJ,A_score,C_score,FSA_Score,FSA_Healthy,ReviewCount_log,weighted_rating,Calories_scaled,FatContent_scaled,SaturatedFatContent_scaled,SodiumContent_scaled,SugarContent_scaled,FiberContent_scaled,ProteinContent_scaled,CarbohydrateContent_scaled,WHO_Score_scaled,FSA_Score_scaled,AggregatedRating_scaled,ReviewCount_log_scaled,CookTime_Minutes_scaled,PrepTime_Minutes_scaled,TotalTime_Minutes_scaled,popularity_score_scaled,weighted_rating_scaled
0,38,Low-Fat Berry Blue Frozen Dessert,1533,Dancer,PT24H,PT45M,PT24H45M,1999-08-09T21:46:00Z,Make and share this Low-Fat Berry Blue Frozen ...,[https://img.sndimg.com/food/image/upload/w_55...,Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Health...","[4, 1/4, 1, 1]","[blueberries, granulated sugar, vanilla yogurt...",4.5,4.0,170.9,2.5,1.3,8.0,29.8,37.1,3.6,30.2,3.2,4.0,NaN,"[Toss 2 cups berries with sugar., Let stand fo...",True,True,4.5,18.0,1440.0,45,1485,1999-08-09 21:46:00+00:00,"[blueberries, granulated sugar, vanilla yogurt...","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]","[blueberry, sugar, vanilla yogurt, lemon juice]",22.5,11.7,148.4,12.8,6.846109,13.165594,True,True,True,3,True,715.0456,9,4,5,False,1.609438,4.544005,-0.224419,-0.198366,-0.177156,-0.175436,0.058349,-0.028274,-0.355593,-0.066303,1.638320,-0.580802,-0.205650,1.055252,0.017742,-0.003585,0.019716,0.064861,-0.386848
1,39,Biryani,1567,elly9812,PT25M,PT4H,PT4H25M,1999-08-29T13:12:00Z,Make and share this Biryani recipe from Food.com.,[https://img.sndimg.com/food/image/upload/w_55...,Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, ...","[1, 4, 2, 2, 8, 1/4, 8, 1/2, 1, 1, 1/4, 1/4, 1...","[saffron, milk, hot green chili peppers, onion...",3.0,1.0,1110.7,58.8,16.6,372.8,368.4,84.4,9.0,20.4,63.4,6.0,NaN,[Soak saffron in warm milk for 5 minutes and p...,True,True,3.0,3.0,25.0,240,265,1999-08-29 13:12:00+00:00,"[saffron, milk, hot green chili p

In [194]:
dataset_summary(recipes_v6)

,dtype,missing_count,missing_pct,n_unique
AggregatedRating,float64,253223,48.46,9
RecipeServings,float64,182911,35.01,171
CookTime_Minutes,float64,82545,15.80,490
CookTime,object,82545,15.80,490
FatPercentEnergy,float64,3111,0.60,332251
SaturatedFatPercentEnergy,float64,3111,0.60,273610
RecipeCategory,object,751,0.14,311
Description,object,5,0.00,492838
Energy_kJ,float64,0,0.00,30138
ingredients_canonical_final_le_5_replace,object,0,0.00,487220


In [207]:
pd.set_option('display.max_rows', None)   # show all rows
pd.set_option('display.width', None)         # no line wrap
pd.set_option('display.max_colwidth', None)  # full text
dataset_summary(recipes_v7)

,dtype,missing_count,missing_pct,n_unique
AggregatedRating,float64,253223,48.46,9
RecipeServings,float64,182911,35.01,171
CookTime_Minutes,float64,82545,15.80,490
CookTime,object,82545,15.80,490
RecipeId,int64,0,0.00,522517
SodiumContent_scaled,float64,0,0.00,40455
SaturatedFatContent_scaled,float64,0,0.00,2533
FatContent_scaled,float64,0,0.00,4523
Calories_scaled,float64,0,0.00,30138
weighted_rating,float64,0,0.00,732


In [ ]:
#recipes_v7.to_csv("recipes_1_cleaned_scaled.csv",index = False)

In [ ]:
def save_dataframe(
    data_frame,
    file_name,
    directory='datasets/raw',
    max_mb=100,
    partition_size='50MB'
):
    df = data_frame.copy()

    # convert pandas string dtype to object to avoid pyarrow dependency in dask
    string_cols = df.select_dtypes(include=["string"]).columns
    if len(string_cols) > 0:
        df[string_cols] = df[string_cols].astype("object")

    if require_batching(df, max_mb=max_mb):
        dask_df = dd.from_pandas(df, npartitions=1)
        dask_df = dask_df.repartition(partition_size=partition_size)
        dask_df.to_csv(directory + '/' + file_name + '*.csv', index=False)
    else:
        df.to_csv(directory + '/' + file_name + '.csv', index=False)

    print('Save data completed.')

In [224]:
from project_package.data_collection.utility import save_dataframe

save_dataframe(
    data_frame=recipes_v7,
    file_name="recipes_1_cleaned_scaled",
    directory="data/processed"
)

ImportError: pyarrow>=10.0.1 is required for PyArrow backed StringArray.

In [223]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.
